 # SET UP

In [ ]:
# --- ติดตั้ง Engine ของ Tesseract ---
!sudo apt install tesseract-ocr
!sudo apt install libtesseract-dev

# --- ติดตั้งไลบรารีที่จำเป็น (หากยังไม่ได้รัน) ---
!pip install pdf2image
!pip install poppler-utils
!pip install numpy
# ใช้ apt-get เพื่อติดตั้ง poppler-utils สำหรับระบบ (Debian/Ubuntu)
!sudo apt-get install -y poppler-utils

# --- ติดตั้ง Python Libraries ---
!pip install pytesseract
!pip install easyocr
!pip install transformers torch sentencepiece
!pip install pdf2image
!pip install poppler-utils # Dependency ของ pdf2image

!pip install openpyxl
!pip install jiwer

# --- 1. ติดตั้งไลบรารีที่จำเป็น ---
!pip install opencv-python-headless numpy matplotlib

In [ ]:
from google.colab import drive
import os
import shutil

mountpoint = '/content/drive'

# Check if the mountpoint exists and is not empty
if os.path.exists(mountpoint) and os.path.isdir(mountpoint):
    if os.listdir(mountpoint):
        print(f"Mountpoint {mountpoint} is not empty. Removing contents...")
        # Remove contents of the directory
        for item in os.listdir(mountpoint):
            item_path = os.path.join(mountpoint, item)
            try:
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.unlink(item_path)
                elif os.path.isdir(item_path):
                    shutil.rmtree(item_path)
            except Exception as e:
                print(f"Error removing {item_path}: {e}")
    else:
        print(f"Mountpoint {mountpoint} is empty.")
elif not os.path.exists(mountpoint):
    print(f"Mountpoint {mountpoint} does not exist. It will be created.")
else:
    print(f"Mountpoint {mountpoint} exists but is not a directory. Please check.")


# Attempt to mount Google Drive
drive.mount(mountpoint)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tran to image

In [ ]:
import os
from pdf2image import convert_from_path
from tqdm import tqdm # เพิ่ม tqdm เพื่อให้เห็น progress bar สวยๆ

# --- 1. ตั้งค่า Path ของโฟลเดอร์ ---
# !!! ให้แก้เป็น path ใน Google Drive ของคุณ !!!
PDF_FOLDER = '/content/drive/MyDrive/lastest-ktep/kmitl_dataset/dataset'
IMAGE_FOLDER = '/content/drive/MyDrive/lastest-ktep/conv-to-img'

# --- 2. สร้างโฟลเดอร์สำหรับเก็บภาพ (หากยังไม่มี) ---
os.makedirs(IMAGE_FOLDER, exist_ok=True)
print(f"ไฟล์รูปภาพจะถูกบันทึกไว้ที่: {IMAGE_FOLDER}")

# --- 3. ดึงรายชื่อไฟล์ PDF ทั้งหมด ---
try:
    # กรองเอาเฉพาะไฟล์ที่ลงท้ายด้วย .pdf หรือ .PDF
    #    files_to_process = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.pdf', '.png', '.jpg', '.jpeg'))]
    pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith('.pdf')]
    if not pdf_files:
        print(f"ไม่พบไฟล์ PDF ในโฟลเดอร์: {PDF_FOLDER}")
    else:
        print(f"พบไฟล์ PDF ทั้งหมด {len(pdf_files)} ไฟล์ กำลังเริ่มทำการแปลง...")

        # --- 4. วนลูปเพื่อแปลงไฟล์ทั้งหมด ---
        for pdf_file in tqdm(pdf_files, desc="Converting PDFs to Images"):
            pdf_path = os.path.join(PDF_FOLDER, pdf_file)

            # ตั้งชื่อไฟล์ output ให้เหมือนไฟล์ input แต่เปลี่ยนนามสกุลเป็น .png
            output_filename = f"{os.path.splitext(pdf_file)[0]}.png"
            image_path = os.path.join(IMAGE_FOLDER, output_filename)

            # ตรวจสอบว่าไฟล์ภาพนี้เคยถูกสร้างไว้แล้วหรือยัง เพื่อจะได้ไม่ต้องทำซ้ำ
            if os.path.exists(image_path):
                # print(f"ข้ามไฟล์ {pdf_file} เนื่องจากมีไฟล์ภาพอยู่แล้ว")
                continue

            try:
                # แปลงไฟล์ PDF เป็นภาพ (เราเอาเฉพาะหน้าแรก)
                images = convert_from_path(pdf_path, dpi=300)
                if images:
                    images[0].save(image_path, 'PNG')
            except Exception as e:
                print(f"เกิดข้อผิดพลาดระหว่างแปลงไฟล์ {pdf_file}: {e}")

        print("\nการแปลงไฟล์ทั้งหมดเสร็จสมบูรณ์!")

except FileNotFoundError:
    print(f"ไม่พบโฟลเดอร์ต้นทาง: {PDF_FOLDER}")
    print("โปรดตรวจสอบว่า Path ถูกต้อง และคุณได้ Mount Google Drive แล้ว")



---



# AUGMENT

**Auto Rotate I'Tong**

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

folder_path = "/content/drive/MyDrive/lastest-ktep/conv-to-img"

for filename in os.listdir(folder_path):
    img_path = os.path.join(folder_path, filename)
    img = cv2.imread(img_path)

    if img is None:
        print(f"Error: Could not load image at {img_path}. Please check the file path.")
    else:
        # สร้างโฟลเดอร์สำหรับบันทึกภาพ
        output_folder = "/content/drive/MyDrive/lastest-ktep/rotate_45"
        os.makedirs(output_folder, exist_ok=True)

        # ขนาดภาพต้นฉบับ
        (h, w) = img.shape[:2]

        # คำนวณขนาดพื้นหลังให้ใหญ่พอรองรับทุกการหมุน
        max_side = int(np.sqrt(w**2 + h**2))  # คำนวณเส้นทแยงมุม
        new_w, new_h = max_side, max_side  # กำหนดขนาดพื้นหลังใหม่

        # ค่ามุมหมุนที่ต้องการ
        rotation_angles = [-45, -30, -15, 15, 30, 45]

        plt.figure(figsize=(12, 6))

        for i, angle in enumerate(rotation_angles):

            # สร้างพื้นหลังสีขาวขนาดใหม่
            padded_img = np.full((new_h, new_w, 3), 255, dtype=np.uint8)  # พื้นหลังสีขาว

            # คำนวณตำแหน่งให้ภาพอยู่กลางพื้นหลัง
            center_x = (new_w - w) // 2
            center_y = (new_h - h) // 2

            # วางภาพต้นฉบับไว้ตรงกลางของพื้นหลัง
            padded_img[center_y:center_y + h, center_x:center_x + w] = img

            # หมุนภาพ
            center = (new_w // 2, new_h // 2)
            rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
            rotated_img = cv2.warpAffine(padded_img, rotation_matrix, (new_w, new_h),
                                        borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255))

            # บันทึกภาพที่หมุนแล้ว
            save_path = os.path.join(output_folder, f'{filename}_rotated_{angle}.png')
            cv2.imwrite(save_path, rotated_img)

            # แสดงภาพ
            plt.subplot(2, 3, i + 1)
            plt.imshow(cv2.cvtColor(rotated_img, cv2.COLOR_BGR2RGB))
            plt.title(f'Rotation: {angle}°')
            plt.axis('off')

        plt.tight_layout()
        plt.show()

        print("✅ เสร็จสิ้น! ภาพทั้งหมดถูกสร้างและบันทึกในโฟลเดอร์:", output_folder)


**Corrected**

In [ ]:
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt

def correct_skew(image, debug_draw=False):
    """แก้ไขการเอียงของภาพโดยใช้ Hough Line Transform"""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)

    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=100, maxLineGap=10)
    angles = []

    if lines is not None:
        if debug_draw:
            debug_img = image.copy()
            for line in lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(debug_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

            plt.figure(figsize=(8, 6))
            plt.title("🔍 Hough Lines (Before Rotation)")
            plt.imshow(cv2.cvtColor(debug_img, cv2.COLOR_BGR2RGB))
            plt.axis('off')
            plt.show()

        for line in lines:
            x1, y1, x2, y2 = line[0]
            angle = np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi
            angles.append(angle)

        median_angle = np.median(angles)
        print(f"🔄 Correcting skew by {median_angle:.2f}°")

        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        rotation_matrix = cv2.getRotationMatrix2D(center, median_angle, 1.0)
        corrected_img = cv2.warpAffine(image, rotation_matrix, (w, h)) ############ borderMode-cv2.BORDER_CONSTANT,

        return corrected_img, median_angle
    else:
        print("⚠️ No lines found for skew correction.")
        return image, 0


def detect_text_region(image):
    """ ตรวจจับตำแหน่งของข้อความหรือขอบบนของเอกสาร """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # แปลงภาพเป็นขาวดำ
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)  # ลด noise ด้วย Gaussian Blur
    edges = cv2.Canny(blurred, 50, 150)  # ตรวจจับขอบภาพ

    # ค้นหา contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None  # ถ้าไม่มี contour ให้คืนค่า None

    # เรียงลำดับ contours ตามตำแหน่ง y (จากบนลงล่าง)
    contours = sorted(contours, key=lambda c: cv2.boundingRect(c)[1])
    return cv2.boundingRect(contours[0])  # คืนค่ากล่องที่อยู่ด้านบนสุด

#cv2.findContours() เป็นฟังก์ชันที่ใช้หาขอบเขตของวัตถุที่เชื่อมต่อกันในภาพ
#cv2.RETR_EXTERNAL จะคืนค่าเฉพาะ contour ที่อยู่รอบนอกสุด ของแต่ละวัตถุในภาพ
#cv2.CHAIN_APPROX_SIMPLE จะลดจำนวนจุดที่ไม่จำเป็นเพื่อลดขนาดของข้อมูล contour ที่ได้

#ตัวแปร contours จะเป็นลิสต์ของเส้นขอบของวัตถุที่พบในภาพ

def is_upside_down(image):
    """ ตรวจสอบว่าภาพกลับหัวหรือไม่โดยใช้การวิเคราะห์ตำแหน่งของข้อความ """
    text_region = detect_text_region(image)

    if text_region is None:
        print("⚠️ No text detected, assuming image is upside down.")
        return True  # ถ้าไม่มีข้อความ ให้ถือว่ากลับหัว

    x, y, w, h = text_region
    print(f"📝 Text detected at y={y}, height={h}")

    img_height = image.shape[0]

    if y > img_height // 2:
        print("❌ Text detected in lower half -> Image is upside down!")
        return True
    else:
        print("✅ Text detected in upper half -> Image is correct!")
        return False

def crop_white_background(image, tol=5):
    """ ตัดพื้นหลังสีขาวออกทั้งหมด โดยใช้ Contour Detection """
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image

    mask = cv2.threshold(gray, 255 - tol, 255, cv2.THRESH_BINARY_INV)[1]  # สร้าง Mask ที่แสดงเฉพาะวัตถุ

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # หา Contours

    if not contours:
        return image  # ถ้าไม่มี contour ให้คืนภาพเดิม

    x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))  # หาขอบเขตของวัตถุที่ใหญ่ที่สุด
    cropped = image[y:y+h, x:x+w]  # ครอบเฉพาะพื้นที่ที่มีวัตถุ

    return cropped

def process_images_in_folder(input_folder, output_folder):
    """ ประมวลผลภาพทั้งหมดในโฟลเดอร์โดยเรียงตามชื่อไฟล์ """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    file_list = sorted(os.listdir(input_folder))  # ✅ เรียงลำดับไฟล์ตามชื่อ

    for filename in file_list:
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        if os.path.isdir(input_path):
            continue

        print(f"📌 Processing: {filename}")
        img = cv2.imread(input_path)

        if img is None:
            print(f"❌ Skipping {filename}: Cannot read image.")
            continue

        corrected_img, angle = correct_skew(img, debug_draw=True)  # ✅ แก้เอียงรูป

        if is_upside_down(corrected_img):  # ✅ ตรวจสอบว่ากลับหัวหรือไม่
            print(f"🔄 Rotating {filename} by 180°")
            corrected_img = cv2.rotate(corrected_img, cv2.ROTATE_180)

        cropped_img = crop_white_background(corrected_img)  # ✅ ตัดขอบขาวออก

        save_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}_corrected.png")
        cv2.imwrite(save_path, cropped_img)  # ✅ บันทึกภาพ
        print(f"✅ Saved corrected image to {save_path}")

        plt.figure(figsize=(6, 6))  # ✅ แสดงผลภาพ
        plt.imshow(cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB))
        plt.title(f"Processed: {filename}")
        plt.axis("off")
        plt.show()

# ระบุโฟลเดอร์ต้นทางและปลายทาง
input_folder = "/content/drive/MyDrive/lastest-ktep/rotate_45"
output_folder = "/content/drive/MyDrive/lastest-ktep/corrected"

# ✅ เรียกใช้งาน
process_images_in_folder(input_folder, output_folder)


**ณุปที่ได้จากตั้งตรง โอเคเลยอะมันตัดขอบมาให้เรียบร้อยเลย
ไม่ต้อง Transparent เอาพื้นหลังสีขาวออกเลย ยังได้คือมันตัดให้ไม่เหลือสีขาวไว้เลย แต่ของ Toeic อาจจะต้องใช้ รึเปล่า ไม่แน่ใจ**

**Transparent  White BG **

In [ ]:
import cv2
import numpy as np
import os
from pathlib import Path

# ======= CONFIG =======
input_folder = "/content/drive/MyDrive/lastest-ktep/corrected"     # โฟลเดอร์ที่มีภาพต้นฉบับ
output_folder = "/content/drive/MyDrive/toeic_dataset/Transparented"       # โฟลเดอร์สำหรับบันทึกผลลัพธ์

# สร้าง output folder ถ้ายังไม่มี
os.makedirs(output_folder, exist_ok=True)

# รูปแบบไฟล์ที่รองรับ
valid_exts = ('.png', '.jpg', '.jpeg')

# วนลูปภาพทั้งหมดใน input folder
for filename in os.listdir(input_folder):
    if filename.lower().endswith(valid_exts):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, Path(filename).stem + ".png")

        # โหลดภาพ
        image = cv2.imread(input_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Threshold เพื่อหาส่วนที่ไม่ใช่พื้นหลังสีขาว
        _, mask = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)

        # หา contour ใหญ่สุด และครอป
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"❌ No content found in {filename}")
            continue

        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        cropped = image[y:y+h, x:x+w]
        mask_cropped = mask[y:y+h, x:x+w]

        # สร้าง alpha channel และรวมเป็น BGRA
        bgra = cv2.cvtColor(cropped, cv2.COLOR_BGR2BGRA)
        bgra[:, :, 3] = mask_cropped

        # บันทึกภาพผลลัพธ์
        cv2.imwrite(output_path, bgra)
        print(f"✅ Processed and saved: {output_path}")

**HSV Black-white**

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# --- 1. ตั้งค่า Path ของโฟลเดอร์ ---
# !!! ให้แก้เป็น path ใน Google Drive ของคุณ !!!

# โฟลเดอร์ที่เก็บภาพสีต้นฉบับ (ที่แปลงจาก PDF)
INPUT_IMAGE_FOLDER = "/content/drive/MyDrive/toeic_dataset/Transparented"

# โฟลเดอร์ที่จะใช้เก็บภาพขาวดำที่ผ่าน Preprocessing แล้ว
OUTPUT_PROCESSED_FOLDER = '/content/drive/MyDrive/lastest-ktep/HSV-B-W'

# --- 2. สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์ (หากยังไม่มี) ---
os.makedirs(OUTPUT_PROCESSED_FOLDER, exist_ok=True)
print(f"รูปภาพที่ประมวลผลแล้วจะถูกบันทึกที่: {OUTPUT_PROCESSED_FOLDER}")


# --- 3. ฟังก์ชัน Preprocessing ที่ดีที่สุดของเรา ---
def create_ocr_ready_image(image_path):
    """
    อ่านภาพสี, ใช้ HSV Mask เพื่อแยกตัวอักษร, และแปลงเป็นภาพขาวดำ
    """
    image = cv2.imread(image_path)
    if image is None:
        return None # กรณีอ่านไฟล์ภาพไม่ได้

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # กำหนดช่วงสีดำและน้ำเงิน
    lower_black = np.array([0, 0, 0])
    upper_black = np.array([180, 255, 120])
    lower_blue = np.array([90, 50, 50])
    upper_blue = np.array([130, 255, 255])

    # สร้างและรวม Mask
    mask_black = cv2.inRange(hsv, lower_black, upper_black)
    mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)
    combined_mask = cv2.bitwise_or(mask_black, mask_blue)

    # สลับสี (Invert) เพื่อให้ตัวอักษรเป็นสีดำ พื้นหลังเป็นสีขาว
    final_binary_image = cv2.bitwise_not(combined_mask)

    return final_binary_image


# --- 4. เริ่มกระบวนการ xử lý ทั้งหมดในโฟลเดอร์ ---
try:
    # กรองเอารายชื่อไฟล์รูปภาพ
    image_files = [f for f in os.listdir(INPUT_IMAGE_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"พบรูปภาพทั้งหมด {len(image_files)} รูป, กำลังเริ่มประมวลผล...")

    # วนลูปเพื่อประมวลผลทีละไฟล์
    for filename in tqdm(image_files, desc="Processing Images"):
        input_path = os.path.join(INPUT_IMAGE_FOLDER, filename)
        output_path = os.path.join(OUTPUT_PROCESSED_FOLDER, filename)

        # ข้ามไฟล์ที่เคยทำไปแล้ว
        if os.path.exists(output_path):
            continue

        try:
            # ประมวลผลภาพ
            ocr_ready_image = create_ocr_ready_image(input_path)
            if ocr_ready_image is not None:
                # บันทึกผลลัพธ์
                cv2.imwrite(output_path, ocr_ready_image)
        except Exception as e:
            print(f"\nเกิดข้อผิดพลาดระหว่างประมวลผลไฟล์ {filename}: {e}")

    print("\nประมวลผลรูปภาพทั้งหมดเสร็จสมบูรณ์!")

except FileNotFoundError:
    print(f"\nERROR: ไม่พบโฟลเดอร์ {INPUT_IMAGE_FOLDER}. กรุณาตรวจสอบ Path อีกครั้ง")

**RESize ก่อนมั้ย**

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
from pathlib import Path

# --- 1. กำหนดค่าและ Path ---
# ⬇️⬇️⬇️ กรุณาแก้ไข Path ทั้งสามส่วนให้ถูกต้อง ⬇️⬇️⬇️

# 1.1 Path ของรูปภาพที่คุณต้องการปรับขนาด
INPUT_IMAGE_PATH = "/content/drive/MyDrive/lastest-ktep/HSV-B-W/KMITL-TEP PILOT-1.png_rotated_-15_corrected.png"
# 1.2 โฟลเดอร์สำหรับเก็บรูปภาพที่ปรับขนาดแล้ว
OUTPUT_FOLDER = "/content/drive/MyDrive/lastest-ktep/resized_output"

# 1.3 ชื่อไฟล์ใหม่สำหรับบันทึก (คุณสามารถเปลี่ยนได้ตามต้องการ)
OUTPUT_FILENAME = "resized_image.png"


# --- 2. กำหนดขนาดสุดท้ายที่ต้องการ ---
FINAL_WIDTH = 2480
FINAL_HEIGHT = 1759


# --- 3. สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์ (หากยังไม่มี) ---
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# --- 4. เริ่มกระบวนการและแสดงผลเปรียบเทียบ ---
try:
    # อ่านภาพต้นฉบับ
    original_image = cv2.imread(INPUT_IMAGE_PATH, cv2.IMREAD_UNCHANGED)
    if original_image is None:
        raise FileNotFoundError(f"ไม่สามารถอ่านไฟล์ภาพได้ที่: {INPUT_IMAGE_PATH}")

    # ดึงขนาดเดิม
    orig_h, orig_w = original_image.shape[:2]

    # ทำการปรับขนาด
    print(f"--- 🚀 Resizing image to {FINAL_WIDTH}x{FINAL_HEIGHT} ---")
    resized_image = cv2.resize(original_image, (FINAL_WIDTH, FINAL_HEIGHT), interpolation=cv2.INTER_LANCZOS4)

    # ⬇️⬇️⬇️ ส่วนที่เพิ่มเข้ามา: บันทึกไฟล์ผลลัพธ์ ⬇️⬇️⬇️
    save_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILENAME)
    cv2.imwrite(save_path, resized_image)
    print(f"--- ✅ Image saved successfully to: {save_path} ---")
    # ⬆️⬆️⬆️ สิ้นสุดส่วนที่เพิ่มเข้ามา ⬆️⬆️⬆️

    # ดึงขนาดใหม่เพื่อแสดงผล
    new_h, new_w = resized_image.shape[:2]

    # --- 5. แสดงผลลัพธ์เปรียบเทียบ ---
    print(f"--- 🖼️ Displaying Comparison ---")
    plt.figure(figsize=(12, 8))

    # ภาพต้นฉบับ
    plt.subplot(1, 2, 1)
    if len(original_image.shape) == 3 and original_image.shape[2] == 3:
        plt.imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
    elif len(original_image.shape) == 3 and original_image.shape[2] == 4:
         plt.imshow(cv2.cvtColor(original_image, cv2.COLOR_BGRA2RGBA))
    else:
        plt.imshow(original_image, cmap='gray')
    plt.title(f"Original Image\nSize: {orig_w} x {orig_h}")
    plt.axis('off')

    # ภาพที่ปรับขนาดแล้ว
    plt.subplot(1, 2, 2)
    if len(resized_image.shape) == 3 and resized_image.shape[2] == 3:
        plt.imshow(cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB))
    elif len(resized_image.shape) == 3 and resized_image.shape[2] == 4:
         plt.imshow(cv2.cvtColor(resized_image, cv2.COLOR_BGRA2RGBA))
    else:
        plt.imshow(resized_image, cmap='gray')
    plt.title(f"Resized Image\nSize: {new_w} x {new_h}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"An error occurred: {e}")

**ทดลองรูปที่ได้**

In [ ]:
import cv2
import pandas as pd
import numpy as np # <<< เพิ่มบรรทัดนี้

import re
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
import pytesseract

# --- 1. กำหนดค่าและ Path ---
# ⬇️⬇️⬇️ ใช้ Path ของภาพที่ผ่าน Preprocess มาดีที่สุดแล้ว ⬇️⬇️⬇️
PROCESSED_IMAGE_PATH = "/content/drive/MyDrive/lastest-ktep/resized_output/resized_image.png"

# พิกัด ROI ที่แน่นอน x= แนวนอน, y= แนวตั้ง, w=กว้าง, h=สูง
ROI_CONFIG = {
    'name': (225, 700, 800, 100), 'application_no': (1675, 700, 500, 100),
    'test_date': (2010, 800, 450, 100), 'overall_score': (1600, 930, 800, 180),
    'grammar': (420, 1225, 500, 120), 'reading': (1600, 1225, 500, 120),
    'speaking': (420, 1410, 500, 120), 'writing': (1600, 1410, 500, 120),
}

# --- 2. ฟังก์ชัน Parse คะแนน (ถ้าต้องการ) ---
def parse_level_and_score(text):
    if not text: return text # คืนค่าเดิมถ้าไม่มีข้อความ
    cleaned_text = text.upper().replace('I', '1').replace('L', '1')
    match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", cleaned_text)
    return f"{match.group(1)}({match.group(2)})" if match else text

# --- 3. เริ่มกระบวนการทดสอบ ---
try:
    # โหลดภาพด้วย PIL
    image_pil = Image.open(PROCESSED_IMAGE_PATH)
    # สร้างภาพสำหรับวาดกรอบ (แปลงเป็นสีเพื่อให้เห็นกรอบสีชัดเจน)
    image_for_drawing = image_pil.convert("RGB")
    image_for_drawing_cv = cv2.cvtColor(np.array(image_for_drawing), cv2.COLOR_RGB2BGR)

    results_data = []

    print("--- 🚀 Starting Boxing OCR Demonstration ---")
    # วนลูปตาม ROI เพื่อสกัดข้อมูลและวาดกรอบ
    for field, (x, y, w, h) in ROI_CONFIG.items():
        x1, y1, x2, y2 = x, y, x + w, y + h
        coords = (x1, y1, x2, y2)

        # ตัดภาพขาวดำเพื่อส่งให้ Tesseract
        cropped_box = image_pil.crop(coords)

        # สกัดข้อความด้วย Tesseract
        config = r'--oem 3 --psm 7' # PSM 7 เหมาะสำหรับหนึ่งบรรทัด
        text = pytesseract.image_to_string(cropped_box, config=config).strip()

        results_data.append({
            "Field": field,
            "Extracted Text": text
        })

        # วาดกรอบและชื่อ Field ลงบนภาพ
        cv2.rectangle(image_for_drawing_cv, (x1, y1), (x2, y2), (0, 255, 0), 3) # สีเขียว
        cv2.putText(image_for_drawing_cv, field, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    print("--- ✅ Extraction Complete ---")

    # --- 4. แสดงผลลัพธ์ ---

    # 4.1 แสดงภาพผลลัพธ์พร้อมกรอบ
    print("\n\n--- 🖼️ Visual Result: Bounding Boxes ---")
    plt.figure(figsize=(15, 20))
    plt.imshow(cv2.cvtColor(image_for_drawing_cv, cv2.COLOR_BGR2RGB))
    plt.title("Bounding Boxes for Each Field", fontsize=20)
    plt.axis('off')
    plt.show()

    # 4.2 แสดงตารางข้อความที่สกัดได้
    print("\n\n--- 📝 Extracted Text Results (Raw) ---")
    results_df = pd.DataFrame(results_data)
    display(results_df)

except FileNotFoundError:
    print(f"\n[ERROR] ไม่พบไฟล์ภาพที่: {PROCESSED_IMAGE_PATH}")
except Exception as e:
    print(f"\nAn error occurred: {e}")

**HSV Auto Resize**

In [ ]:
import cv2
import os
from tqdm import tqdm
from pathlib import Path

# --- 1. กำหนดค่าและ Path ---
# ⬇️⬇️⬇️ กรุณาแก้ไข Path ให้ถูกต้อง ⬇️⬇️⬇️

# โฟลเดอร์ที่เก็บรูปภาพที่คุณต้องการปรับขนาด
INPUT_FOLDER = "/content/drive/MyDrive/lastest-ktep/HSV-B-W"

# โฟลเดอร์สำหรับเก็บรูปภาพที่ปรับขนาดแล้ว
OUTPUT_FOLDER = "/content/drive/MyDrive/lastest-ktep/HSV_resized"

# --- 2. กำหนดขนาดสุดท้ายที่ต้องการ ---
FINAL_WIDTH = 2480
FINAL_HEIGHT = 1759

# --- 3. สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์ (หากยังไม่มี) ---
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"รูปภาพที่ปรับขนาดแล้วจะถูกบันทึกที่: {OUTPUT_FOLDER}")


# --- 4. เริ่มกระบวนการปรับขนาดทั้งหมดในโฟลเดอร์ ---
def resize_images_in_folder(input_dir, output_dir, target_width, target_height):

    # ดึงรายชื่อไฟล์ภาพทั้งหมด
    try:
        image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not image_files:
            print(f"ไม่พบไฟล์รูปภาพในโฟลเดอร์: {input_dir}")
            return
    except FileNotFoundError:
        print(f"[ERROR] ไม่พบโฟลเดอร์ต้นทาง: {input_dir}")
        return

    print(f"พบรูปภาพทั้งหมด {len(image_files)} รูป, กำลังเริ่มปรับขนาด...")

    # วนลูปเพื่อประมวลผลทีละไฟล์
    for filename in tqdm(image_files, desc="Resizing Images"):
        input_path = os.path.join(input_dir, filename)

        # สร้างชื่อไฟล์ใหม่ในโฟลเดอร์ปลายทาง
        # ใช้ Path(filename).stem เพื่อเอาเฉพาะชื่อไฟล์ไม่รวมนามสกุล
        save_path = os.path.join(output_dir, Path(filename).stem + ".png") # บันทึกเป็น PNG เพื่อคุณภาพดีที่สุด

        # ข้ามไฟล์ที่เคยทำไปแล้ว
        if os.path.exists(save_path):
            continue

        try:
            # อ่านภาพ
            image = cv2.imread(input_path, cv2.IMREAD_UNCHANGED) # IMREAD_UNCHANGED เพื่อรักษา Alpha channel
            if image is None:
                print(f"Warning: ไม่สามารถอ่านไฟล์ {filename}, ข้ามไฟล์นี้")
                continue

            # ปรับขนาดภาพด้วยอัลกอริทึม INTER_LANCZOS4 ซึ่งให้คุณภาพดีที่สุด
            resized_image = cv2.resize(image, (target_width, target_height), interpolation=cv2.INTER_LANCZOS4)

            # บันทึกภาพที่ปรับขนาดแล้ว
            cv2.imwrite(save_path, resized_image)

        except Exception as e:
            print(f"\nเกิดข้อผิดพลาดระหว่างประมวลผลไฟล์ {filename}: {e}")

    print("\n--- ✅ ปรับขนาดรูปภาพทั้งหมดเสร็จสมบูรณ์! ---")


# --- เรียกใช้งานฟังก์ชัน ---
resize_images_in_folder(INPUT_FOLDER, OUTPUT_FOLDER, FINAL_WIDTH, FINAL_HEIGHT)

# Preproceess image แบบตั้งตรง

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from PIL import Image
import matplotlib.pyplot as plt

# --- 1. โหลดรูปภาพ ---
# !!! แก้ไข Path ให้ถูกต้อง !!!
IMAGE_PATH = '/content/drive/MyDrive/lastest-ktep/aug_transparent/KMITL-TEP PILOT-1.png_rotated_-30.png'
image = cv2.imread(IMAGE_PATH)

# --- 2. สร้าง MASK จากสี (โค้ดส่วนใหญ่ของคุณ) ---
# แปลงภาพเป็น HSV
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

# กำหนดช่วงสีดำ (ปรับค่า upper_black เล็กน้อยเพื่อความแม่นยำ)
lower_black = np.array([0, 0, 0])
upper_black = np.array([180, 255, 120]) # เพิ่มช่วง Hue และ Saturation, ลด Value

# กำหนดช่วงสีน้ำเงิน
lower_blue = np.array([90, 50, 50])
upper_blue = np.array([130, 255, 255])

# สร้าง Mask สำหรับแต่ละสี
mask_black = cv2.inRange(hsv, lower_black, upper_black)
mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

# รวม Mask ทั้งสองเข้าด้วยกัน
# สิ่งใดที่เป็นสีดำหรือสีน้ำเงิน จะกลายเป็นสีขาว (ค่า 255) ใน combined_mask
combined_mask = cv2.bitwise_or(mask_black, mask_blue)


# --- 3. ขั้นตอนสำคัญ: เปลี่ยน MASK ให้เป็นภาพสำหรับ OCR ---
# ใน combined_mask ตอนนี้ ตัวอักษรเป็นสีขาว พื้นหลังเป็นสีดำ
# OCR ต้องการให้ตัวอักษรเป็นสีดำ และพื้นหลังเป็นสีขาว เราจึงต้อง "สลับสี" (Invert)
# cv2.bitwise_not จะเปลี่ยนขาวเป็นดำ และดำเป็นขาว
final_binary_image = cv2.bitwise_not(combined_mask)


# --- 4. แสดงผลลัพธ์เพื่อเปรียบเทียบ ---
plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('1. Original Image')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(combined_mask, cmap='gray')
plt.title('2. Your Combined Mask\n(Text is White)')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(final_binary_image, cmap='gray')
plt.title('3. Final OCR-Ready Image\n(Text is Black)')
plt.axis('off')

plt.show()

# บันทึกภาพผลลัพธ์สุดท้าย
output_path = 'ocr_ready_output.png'
cv2.imwrite(output_path, final_binary_image)
print(f"บันทึกภาพที่พร้อมสำหรับ OCR แล้วที่: {output_path}")

**AUTO Black-white**

In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# --- 1. ตั้งค่า Path ของโฟลเดอร์ ---
# !!! ให้แก้เป็น path ใน Google Drive ของคุณ !!!

# โฟลเดอร์ที่เก็บภาพสีต้นฉบับ (ที่แปลงจาก PDF)
INPUT_IMAGE_FOLDER = '/content/drive/MyDrive/lastest-ktep/conv-to-img'

# โฟลเดอร์ที่จะใช้เก็บภาพขาวดำที่ผ่าน Preprocessing แล้ว
OUTPUT_PROCESSED_FOLDER = '/content/drive/MyDrive/lastest-ktep/prep-img/'

# --- 2. สร้างโฟลเดอร์สำหรับเก็บผลลัพธ์ (หากยังไม่มี) ---
os.makedirs(OUTPUT_PROCESSED_FOLDER, exist_ok=True)
print(f"รูปภาพที่ประมวลผลแล้วจะถูกบันทึกที่: {OUTPUT_PROCESSED_FOLDER}")


# --- 3. ฟังก์ชัน Preprocessing ที่ดีที่สุดของเรา ---
def create_ocr_ready_image(image_path):
    """
    อ่านภาพสี, ใช้ HSV Mask เพื่อแยกตัวอักษร, และแปลงเป็นภาพขาวดำ
    """
    image = cv2.imread(image_path)
    if image is None:
        return None # กรณีอ่านไฟล์ภาพไม่ได้

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # กำหนดช่วงสีดำและน้ำเงิน
    lower_black = np.array([0, 0, 0])
    upper_black = np.array([180, 255, 120])
    lower_blue = np.array([90, 50, 50])
    upper_blue = np.array([130, 255, 255])

    # สร้างและรวม Mask
    mask_black = cv2.inRange(hsv, lower_black, upper_black)
    mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)
    combined_mask = cv2.bitwise_or(mask_black, mask_blue)

    # สลับสี (Invert) เพื่อให้ตัวอักษรเป็นสีดำ พื้นหลังเป็นสีขาว
    final_binary_image = cv2.bitwise_not(combined_mask)

    return final_binary_image


# --- 4. เริ่มกระบวนการ xử lý ทั้งหมดในโฟลเดอร์ ---
try:
    # กรองเอารายชื่อไฟล์รูปภาพ
    image_files = [f for f in os.listdir(INPUT_IMAGE_FOLDER) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    print(f"พบรูปภาพทั้งหมด {len(image_files)} รูป, กำลังเริ่มประมวลผล...")

    # วนลูปเพื่อประมวลผลทีละไฟล์
    for filename in tqdm(image_files, desc="Processing Images"):
        input_path = os.path.join(INPUT_IMAGE_FOLDER, filename)
        output_path = os.path.join(OUTPUT_PROCESSED_FOLDER, filename)

        # ข้ามไฟล์ที่เคยทำไปแล้ว
        if os.path.exists(output_path):
            continue

        try:
            # ประมวลผลภาพ
            ocr_ready_image = create_ocr_ready_image(input_path)
            if ocr_ready_image is not None:
                # บันทึกผลลัพธ์
                cv2.imwrite(output_path, ocr_ready_image)
        except Exception as e:
            print(f"\nเกิดข้อผิดพลาดระหว่างประมวลผลไฟล์ {filename}: {e}")

    print("\nประมวลผลรูปภาพทั้งหมดเสร็จสมบูรณ์!")

except FileNotFoundError:
    print(f"\nERROR: ไม่พบโฟลเดอร์ {INPUT_IMAGE_FOLDER}. กรุณาตรวจสอบ Path อีกครั้ง")



---



# pytesseract


In [ ]:
import pytesseract
from PIL import Image

# --- 1. กำหนดค่าและ Path ---
# ระบุ Path ของภาพที่ผ่าน Preprocess แล้ว (ไฟล์ขาวดำ)
# ผมจะอิงจาก Path ที่คุณเคยใช้ล่าสุด และเลือกไฟล์ภาพที่ 1 มาทดสอบ
IMAGE_PATH = "/content/drive/MyDrive/lastest-ktep/HSV_resized/KMITL-TEP PILOT-1.png_rotated_-30_corrected.png"

# --- 2. เริ่มกระบวนการสกัดข้อความดิบ ---
try:
    print(f"--- 🚀 กำลังอ่านข้อความจากรูปภาพ: {IMAGE_PATH} ---")

    # ใช้ Tesseract อ่านทั้งหน้า
    # Page Segmentation Mode (psm) 4: จะพยายามมองหาข้อความที่เป็นคอลัมน์เดียว
    config = r'--oem 3 --psm 4'

    # สกัดข้อความดิบ
    raw_text = pytesseract.image_to_string(IMAGE_PATH, config=config)

    # --- 3. แสดงผลลัพธ์ ---
    print("\n" + "="*50)
    print("📄 RAW TEXT OUTPUT FROM TESSERACT (PURE OCR) 📄")
    print("="*50)
    # พิมพ์ผลลัพธ์ที่ได้ออกมาทั้งหมด
    print(raw_text)
    print("="*50)
    print("\n--- ✅ เสร็จสิ้น ---")
    print("โปรดตรวจสอบข้อความด้านบนเพื่อหา Pattern สำหรับขั้นตอนถัดไป")

except FileNotFoundError:
    print(f"\n[ERROR] ไม่พบไฟล์ที่ Path: {IMAGE_PATH}")
    print("กรุณาตรวจสอบว่า Path ของไฟล์ถูกต้องหรือไม่")
except Exception as e:
    print(f"\nAn error occurred: {e}")

แก้ไขฟังชั่น ให้ Grammar B1 (17) Reading _B1 (17)
Speaking Bl] (18) Writing B1 (16) ออกมาถูกต้อง

In [ ]:
import cv2
import pandas as pd
import re
import pytesseract
from PIL import Image
from IPython.display import display

# --- 1. กำหนดค่าและ Path ---
ORIGINAL_IMAGE_PATH = "/content/drive/MyDrive/lastest-ktep/prep-img/KMITL-TEP PILOT-1.png"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"
TARGET_ID = 1

# --- 2. Parser ฉบับสมบูรณ์ที่สุด ---
def final_parser(full_text):
    data = {}

    def clean_level(level_text):
        if not level_text: return None
        cleaned = level_text.upper().replace('I', '1').replace('l', '1')
        if 'B11' in cleaned: return 'B1'
        return cleaned

    def parse_score(text):
        if not text: return None, None
        match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", text)
        return (clean_level(match.group(1)), match.group(2)) if match else (None, None)

    # Header
    name_match = re.search(r"Name\s*:?\s*(.*?)\s*(?:Application No|Date of test)", full_text, re.DOTALL | re.IGNORECASE)
    data['name'] = re.sub(r'\s+', ' ', name_match.group(1).strip()) if name_match else None
    app_no_match = re.search(r"Application No\.?\s*:*\s*([A-Z0-9\-]+)", full_text, re.IGNORECASE)
    data['application_no'] = app_no_match.group(1).strip() if app_no_match else None

    # ⬇️⬇️⬇️ แก้ไข Regex ของ Test Date ที่นี่ ⬇️⬇️⬇️
    date_match = re.search(r"Administration\s*:?\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", full_text, re.IGNORECASE)
    data['test_date'] = date_match.group(1).strip() if date_match else None

    # Scores
    total_block = re.search(r"KMITL-TEP Level(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['total_level'], data['total_score'] = parse_score(total_block.group(1) if total_block else "")
    grammar_block = re.search(r"Grammar(.*?)Reading", full_text, re.DOTALL | re.IGNORECASE)
    data['grammar_level'], data['grammar_score'] = parse_score(grammar_block.group(1) if grammar_block else "")
    reading_block = re.search(r"Reading(.*?)Speaking", full_text, re.DOTALL | re.IGNORECASE)
    data['reading_level'], data['reading_score'] = parse_score(reading_block.group(1) if reading_block else "")
    speaking_block = re.search(r"Speaking(.*?)Writing", full_text, re.DOTALL | re.IGNORECASE)
    data['speaking_level'], data['speaking_score'] = parse_score(speaking_block.group(1) if speaking_block else "")
    writing_block = re.search(r"Writing(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['writing_level'], data['writing_score'] = parse_score(writing_block.group(1) if writing_block else "")

    return data

# --- 3. ฟังก์ชัน Preprocessing ---
def create_ocr_ready_image(image_path):
    image = cv2.imread(image_path)
    if image is None: return None
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    binary_image = cv2.adaptiveThreshold(gray_image, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    return binary_image

# --- 4. เริ่มกระบวนการทั้งหมด ---
try:
    # โหลด Ground Truth
    gt_df = pd.read_excel(GROUND_TRUTH_PATH)
    target_gt_row = gt_df[gt_df['No.'] == TARGET_ID].iloc[0]

    # ประมวลผลและสกัดข้อมูล
    processed_image = create_ocr_ready_image(ORIGINAL_IMAGE_PATH)
    full_text = pytesseract.image_to_string(processed_image, config=r'--oem 3 --psm 3')
    extracted_data = final_parser(full_text)

    # สร้างตารางเปรียบเทียบ
    comparison_data = [
        {'Field': 'Name', 'Ground Truth': target_gt_row['Name'], 'Prediction': extracted_data.get('name')},
        {'Field': 'Application No.', 'Ground Truth': target_gt_row['Application No.'], 'Prediction': extracted_data.get('application_no')},
        {'Field': 'Test Date', 'Ground Truth': target_gt_row['Test Date'], 'Prediction': extracted_data.get('test_date')},
        {'Field': 'Grammar Level', 'Ground Truth': target_gt_row['Grammar_Level'], 'Prediction': extracted_data.get('grammar_level')},
        {'Field': 'Grammar Score', 'Ground Truth': target_gt_row['Grammar_Score'], 'Prediction': extracted_data.get('grammar_score')},
        {'Field': 'Reading Level', 'Ground Truth': target_gt_row['Reading_Level'], 'Prediction': extracted_data.get('reading_level')},
        {'Field': 'Reading Score', 'Ground Truth': target_gt_row['Reading_Score'], 'Prediction': extracted_data.get('reading_score')},
        {'Field': 'Speaking Level', 'Ground Truth': target_gt_row['Speaking_Level'], 'Prediction': extracted_data.get('speaking_level')},
        {'Field': 'Speaking Score', 'Ground Truth': target_gt_row['Speaking_Score'], 'Prediction': extracted_data.get('speaking_score')},
        {'Field': 'Writing Level', 'Ground Truth': target_gt_row['Writing_Level'], 'Prediction': extracted_data.get('writing_level')},
        {'Field': 'Writing Score', 'Ground Truth': target_gt_row['Writing_Score'], 'Prediction': extracted_data.get('writing_score')},
        {'Field': 'Total Level', 'Ground Truth': target_gt_row['Total_Level'], 'Prediction': extracted_data.get('total_level')},
        {'Field': 'Total Score', 'Ground Truth': target_gt_row['Total_Score'], 'Prediction': extracted_data.get('total_score')},
    ]
    comparison_df = pd.DataFrame(comparison_data)

    # แสดงผลลัพธ์
    print("\n--- ✅ Final Result: Tesseract (Pure OCR) vs Ground Truth (Corrected) ---")
    display(comparison_df)

except Exception as e:
    print(f"\nAn error occurred: {e}")

In [ ]:
import cv2
import pandas as pd
import os
import re
import jiwer
import numpy as np
import pytesseract
from tqdm import tqdm

# --- 1. กำหนดค่าและ Path ---
MODEL_NAME = 'Tesseract'
APPROACH_NAME = 'Pure OCR'
MASTER_OUTPUT_PATH = "/content/drive/MyDrive/lastest-ktep/excel/comparison_report.xlsx"
INDIVIDUAL_REPORT_PATH = f"/content/drive/MyDrive/lastest-ktep/excel/{MODEL_NAME.lower()}_{APPROACH_NAME.replace(' ', '_').lower()}_report.xlsx"
IMAGE_DIR = "/content/drive/MyDrive/lastest-ktep/prep-img"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"

# --- 2. ฟังก์ชันช่วยเหลือ ---
# (ไม่มีการเปลี่ยนแปลงในส่วนฟังก์ชัน)
def get_id_from_filename(filename):
    match = re.search(r'(\d+)\.(png|jpg|jpeg)$', filename.lower())
    if match: return int(match.group(1))
    return None

def normalize_text(text):
    if text is None: return ""
    text_str = str(text).strip().lower().replace(' ', '')
    if text_str.endswith('.0'): text_str = text_str[:-2]
    return text_str

def final_robust_parser(full_text):
    data = {}
    def clean_level(level_text):
        if not level_text: return None
        cleaned = level_text.upper().replace('I', '1').replace('l', '1')
        if 'B11' in cleaned: return 'B1'
        return cleaned
    def parse_score(text):
        if not text: return None, None
        match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", text)
        return (clean_level(match.group(1)), match.group(2)) if match else (None, None)
    name_match = re.search(r"Name\s*:?\s*(.*?)\s*(?:Application No|Date of test)", full_text, re.DOTALL | re.IGNORECASE)
    data['name'] = re.sub(r'\s+', ' ', name_match.group(1).strip()) if name_match else None
    app_no_match = re.search(r"Application No\.?\s*:*\s*([A-Z0-9\-]+)", full_text, re.IGNORECASE)
    data['application_no'] = app_no_match.group(1).strip() if app_no_match else None
    date_match = re.search(r"Administration\s*:?\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", full_text, re.IGNORECASE)
    data['test_date'] = date_match.group(1).strip() if date_match else None
    total_block = re.search(r"KMITL-TEP Level(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['total_level'], data['total_score'] = parse_score(total_block.group(1) if total_block else "")
    grammar_block = re.search(r"Grammar(.*?)Reading", full_text, re.DOTALL | re.IGNORECASE)
    data['grammar_level'], data['grammar_score'] = parse_score(grammar_block.group(1) if grammar_block else "")
    reading_block = re.search(r"Reading(.*?)Speaking", full_text, re.DOTALL | re.IGNORECASE)
    data['reading_level'], data['reading_score'] = parse_score(reading_block.group(1) if reading_block else "")
    speaking_block = re.search(r"Speaking(.*?)Writing", full_text, re.DOTALL | re.IGNORECASE)
    data['speaking_level'], data['speaking_score'] = parse_score(speaking_block.group(1) if speaking_block else "")
    writing_block = re.search(r"Writing(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['writing_level'], data['writing_score'] = parse_score(writing_block.group(1) if writing_block else "")
    return data

# --- 3. โหลดและตรวจสอบข้อมูลก่อนเริ่ม ---
print("--- 🔍 ขั้นตอนตรวจสอบข้อมูล ---")
# ⬇️⬇️⬇️ แก้ไขการโหลด Excel ที่นี่ ⬇️⬇️⬇️
gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
# ⬆️⬆️⬆️ สิ้นสุดส่วนที่แก้ไข ⬆️⬆️⬆️
gt_df['No.'] = gt_df['No.'].astype(int)
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))], key=lambda f: get_id_from_filename(f) or 0)
print(f"พบ ID ใน Ground Truth ทั้งหมด: {len(gt_df['No.'].unique())} ID | พบไฟล์รูปภาพ: {len(image_files)} ไฟล์")
print("--- ✅ ตรวจสอบข้อมูลเสร็จสิ้น ---\n")

# --- 4. เริ่มกระบวนการสกัดข้อมูล ---
results_list = []
print(f"--- 🚀 เริ่มการสกัดข้อมูลจากรูปภาพ {len(image_files)} ไฟล์ ด้วย {MODEL_NAME} ({APPROACH_NAME}) ---")
for filename in tqdm(image_files, desc=f"Processing with {MODEL_NAME}"):
    image_path = os.path.join(IMAGE_DIR, filename)
    image_id = get_id_from_filename(filename)
    if image_id is None: continue

    gt_row_df = gt_df[gt_df['No.'] == image_id]
    if gt_row_df.empty: continue
    gt_row = gt_row_df.iloc[0]

    try:
        config = r'--oem 3 --psm 3'
        full_text = pytesseract.image_to_string(image_path, config=config)
        extracted_data = final_robust_parser(full_text)
    except Exception as ocr_error:
        print(f"Error processing file {filename}: {ocr_error}")
        continue

    results_list.append({
        "No.": image_id,
        "Application No. (GT)": gt_row["Application No."], "Application No. (Pred)": extracted_data.get('application_no'),
        "Name (GT)": gt_row["Name"], "Name (Pred)": extracted_data.get('name'),
        "Test Date (GT)": gt_row["Test Date"], "Test Date (Pred)": extracted_data.get('test_date'),
        "Grammar_Level (GT)": gt_row["Grammar_Level"], "Grammar_Level (Pred)": extracted_data.get('grammar_level'),
        "Grammar_Score (GT)": gt_row["Grammar_Score"], "Grammar_Score (Pred)": extracted_data.get('grammar_score'),
        "Reading_Level (GT)": gt_row["Reading_Level"], "Reading_Level (Pred)": extracted_data.get('reading_level'),
        "Reading_Score (GT)": gt_row["Reading_Score"], "Reading_Score (Pred)": extracted_data.get('reading_score'),
        "Speaking_Level (GT)": gt_row["Speaking_Level"], "Speaking_Level (Pred)": extracted_data.get('speaking_level'),
        "Speaking_Score (GT)": gt_row["Speaking_Score"], "Speaking_Score (Pred)": extracted_data.get('speaking_score'),
        "Writing_Level (GT)": gt_row["Writing_Level"], "Writing_Level (Pred)": extracted_data.get('writing_level'),
        "Writing_Score (GT)": gt_row["Writing_Score"], "Writing_Score (Pred)": extracted_data.get('writing_score'),
        "Total_Level (GT)": gt_row["Total_Level"], "Total_Level (Pred)": extracted_data.get('total_level'),
        "Total_Score (GT)": gt_row["Total_Score"], "Total_Score (Pred)": extracted_data.get('total_score'),
    })

ocr_results_df = pd.DataFrame(results_list)
print("--- ✅ สกัดข้อมูลเสร็จสิ้น ---\n")

# --- 5 & 6. การประเมินผลและบันทึก (ไม่มีการเปลี่ยนแปลง) ---
print("--- 📊 เริ่มการประเมินผล ---")
eval_df = ocr_results_df.fillna('')
fields_to_evaluate = {
    'Name': ('Name (GT)', 'Name (Pred)'), 'Application No.': ('Application No. (GT)', 'Application No. (Pred)'),
    'Test Date': ('Test Date (GT)', 'Test Date (Pred)'), 'Grammar_Level': ('Grammar_Level (GT)', 'Grammar_Level (Pred)'),
    'Grammar_Score': ('Grammar_Score (GT)', 'Grammar_Score (Pred)'), 'Reading_Level': ('Reading_Level (GT)', 'Reading_Level (Pred)'),
    'Reading_Score': ('Reading_Score (GT)', 'Reading_Score (Pred)'), 'Speaking_Level': ('Speaking_Level (GT)', 'Speaking_Level (Pred)'),
    'Speaking_Score': ('Speaking_Score (GT)', 'Speaking_Score (Pred)'), 'Writing_Level': ('Writing_Level (GT)', 'Writing_Level (Pred)'),
    'Writing_Score': ('Writing_Score (GT)', 'Writing_Score (Pred)'), 'Total_Level': ('Total_Level (GT)', 'Total_Level (Pred)'),
    'Total_Score': ('Total_Score (GT)', 'Total_Score (Pred)'),
}
evaluation_summary_list = []
for field, (gt_col, pred_col) in fields_to_evaluate.items():
    ground_truth = [normalize_text(t) for t in eval_df[gt_col]]
    prediction = [normalize_text(t) for t in eval_df[pred_col]]
    accuracy = np.mean([1 if gt == pred else 0 for gt, pred in zip(ground_truth, prediction)]) * 100
    try:
        error_metrics = jiwer.compute_measures(ground_truth, prediction)
        wer = error_metrics.get('wer', 0) * 100
        cer = error_metrics.get('cer', 0) * 100
        H = error_metrics.get('hits', 0); I = error_metrics.get('insertions', 0); D = error_metrics.get('deletions', 0); S = error_metrics.get('substitutions', 0)
        precision = H / (H + I + S) if (H + I + S) > 0 else 0
        recall = H / (H + D + S) if (H + D + S) > 0 else 0
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    except ValueError:
        wer, cer, f1_score = 100, 100, 0
    evaluation_summary_list.append({
        'Field': field, 'Accuracy (%)': round(accuracy, 2), 'WER (%)': round(wer, 2),
        'CER (%)': round(cer, 2), 'F1-score (%)': round(f1_score * 100, 2)
    })
eval_summary_df = pd.DataFrame(evaluation_summary_list)
eval_summary_df.insert(0, 'Approach', APPROACH_NAME)
eval_summary_df.insert(0, 'Model', MODEL_NAME)
print(eval_summary_df.to_string(index=False))
print("--- ✅ ประเมินผลเสร็จสิ้น ---\n")

# --- 6. บันทึกผลลัพธ์ ---
print(f"--- 💾 กำลังบันทึกรายงานเฉพาะของ {MODEL_NAME} ({APPROACH_NAME}) ---")
with pd.ExcelWriter(INDIVIDUAL_REPORT_PATH, engine='openpyxl') as writer:
    ocr_results_df.to_excel(writer, sheet_name='Detailed_Results', index=False)
    eval_summary_df.drop(columns=['Model', 'Approach']).to_excel(writer, sheet_name='Evaluation_Summary', index=False)
print(f"🎉 บันทึกไฟล์ {INDIVIDUAL_REPORT_PATH} สำเร็จ!")

print(f"--- 💾 กำลังอัปเดตไฟล์ Master Report: {MASTER_OUTPUT_PATH} ---")
SHEET_NAME = 'Master_Evaluation'
try:
    master_df = pd.read_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME)
    master_df = master_df[~((master_df['Model'] == MODEL_NAME) & (master_df['Approach'] == APPROACH_NAME))]
    combined_df = pd.concat([master_df, eval_summary_df], ignore_index=True)
except FileNotFoundError:
    print(f"ไม่พบไฟล์ Master เดิม, กำลังสร้างไฟล์ใหม่...")
    combined_df = eval_summary_df
combined_df.to_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME, index=False)
print(f"🎉 อัปเดตไฟล์ Master Report สำเร็จ!")

**Boxing**

In [ ]:

import cv2
import pandas as pd
import os
import re
import jiwer
import numpy as np
import pytesseract
from tqdm import tqdm
from PIL import Image

# --- 1. กำหนดค่าและ Path ที่สำคัญ ---
MODEL_NAME = 'Tesseract'
APPROACH_NAME = 'Boxing OCR' # <<< ยืนยันใช้แนวทางที่ดีที่สุด
MASTER_OUTPUT_PATH = "/content/drive/MyDrive/lastest-ktep/excel/comparison_report.xlsx"
INDIVIDUAL_REPORT_PATH = f"/content/drive/MyDrive/lastest-ktep/excel/{MODEL_NAME.lower()}_{APPROACH_NAME.replace(' ', '_').lower()}_report.xlsx"
IMAGE_DIR = "/content/drive/MyDrive/lastest-ktep/prep-img" # โฟลเดอร์ที่เก็บภาพที่ Process แล้ว
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"

# พิกัด ROI ที่แน่นอน
ROI_CONFIG = {
    'name': (200, 485, 600, 65), 'application_no': (1100, 485, 300, 60),
    'test_date': (1310, 545, 300, 60), 'overall_score': (820, 625, 800, 100),
    'grammar': (320, 785, 500, 80), 'reading': (1050, 785, 500, 80),
    'speaking': (320, 900, 500, 80), 'writing': (1050, 900, 500, 80),
}

# --- 2. ฟังก์ชันช่วยเหลือ ---
def get_id_from_filename(filename):
    match = re.search(r'(\d+)\.(png|jpg|jpeg)$', filename.lower())
    if match: return int(match.group(1))
    return None

def normalize_text(text):
    if text is None: return ""
    text_str = str(text).strip().lower().replace(' ', '')
    if text_str.endswith('.0'): text_str = text_str[:-2]
    return text_str

def parse_level_and_score(text):
    if not text: return None, None
    match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", text.upper().replace('I','1').replace('l','1'))
    return (match.group(1), match.group(2)) if match else (None, None)

# --- 3. โหลดและตรวจสอบข้อมูลก่อนเริ่ม ---
print("--- 🔍 ขั้นตอนตรวจสอบข้อมูล ---")
gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
gt_df['No.'] = gt_df['No.'].astype(int)
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))], key=lambda f: get_id_from_filename(f) or 0)
print(f"พบ ID ใน Ground Truth ทั้งหมด: {len(gt_df['No.'].unique())} ID | พบไฟล์รูปภาพ: {len(image_files)} ไฟล์")
print("--- ✅ ตรวจสอบข้อมูลเสร็จสิ้น ---\n")

# --- 4. เริ่มกระบวนการสกัดข้อมูล ---
results_list = []
print(f"--- 🚀 เริ่มการสกัดข้อมูลจากรูปภาพ {len(image_files)} ไฟล์ ด้วย {MODEL_NAME} ({APPROACH_NAME}) ---")
for filename in tqdm(image_files, desc=f"Processing with {MODEL_NAME}"):
    image_id = get_id_from_filename(filename)
    if image_id is None: continue

    gt_row_df = gt_df[gt_df['No.'] == image_id]
    if gt_row_df.empty: continue
    gt_row = gt_row_df.iloc[0]

    image_path = os.path.join(IMAGE_DIR, filename)

    try:
        processed_image_pil = Image.open(image_path)
        extracted_data = {}
        for field, (x, y, w, h) in ROI_CONFIG.items():
            x1, y1, x2, y2 = x, y, x + w, y + h
            cropped_box = processed_image_pil.crop((x1, y1, x2, y2))
            config = r'--oem 3 --psm 7'
            text = pytesseract.image_to_string(cropped_box, config=config).strip()

            if field in ['grammar', 'reading', 'speaking', 'writing', 'overall_score']:
                level, score = parse_level_and_score(text)
                if field == 'overall_score':
                    extracted_data['total_level'], extracted_data['total_score'] = level, score
                else:
                    extracted_data[f'{field}_level'], extracted_data[f'{field}_score'] = level, score
            else:
                extracted_data[field] = text
    except Exception as ocr_error:
        print(f"Error processing file {filename}: {ocr_error}")
        continue

    results_list.append({
        "No.": image_id,
        "Application No. (GT)": gt_row["Application No."], "Application No. (Pred)": extracted_data.get('application_no'),
        "Name (GT)": gt_row["Name"], "Name (Pred)": extracted_data.get('name'),
        "Test Date (GT)": gt_row["Test Date"], "Test Date (Pred)": extracted_data.get('test_date'),
        "Grammar_Level (GT)": gt_row["Grammar_Level"], "Grammar_Level (Pred)": extracted_data.get('grammar_level'),
        "Grammar_Score (GT)": gt_row["Grammar_Score"], "Grammar_Score (Pred)": extracted_data.get('grammar_score'),
        "Reading_Level (GT)": gt_row["Reading_Level"], "Reading_Level (Pred)": extracted_data.get('reading_level'),
        "Reading_Score (GT)": gt_row["Reading_Score"], "Reading_Score (Pred)": extracted_data.get('reading_score'),
        "Speaking_Level (GT)": gt_row["Speaking_Level"], "Speaking_Level (Pred)": extracted_data.get('speaking_level'),
        "Speaking_Score (GT)": gt_row["Speaking_Score"], "Speaking_Score (Pred)": extracted_data.get('speaking_score'),
        "Writing_Level (GT)": gt_row["Writing_Level"], "Writing_Level (Pred)": extracted_data.get('writing_level'),
        "Writing_Score (GT)": gt_row["Writing_Score"], "Writing_Score (Pred)": extracted_data.get('writing_score'),
        "Total_Level (GT)": gt_row["Total_Level"], "Total_Level (Pred)": extracted_data.get('total_level'),
        "Total_Score (GT)": gt_row["Total_Score"], "Total_Score (Pred)": extracted_data.get('total_score'),
    })

ocr_results_df = pd.DataFrame(results_list)
print("--- ✅ สกัดข้อมูลเสร็จสิ้น ---\n")

# --- 5 & 6. การประเมินผลและบันทึก (เหมือนเดิม) ---
# ... (วางโค้ดส่วนที่ 5 และ 6 ของคุณต่อจากนี้ได้เลย) ...

# --- 5. เริ่มกระบวนการประเมินผล (Evaluation) ---
print("--- 📊 เริ่มการประเมินผล ---")
eval_df = ocr_results_df.fillna('')
fields_to_evaluate = {
    'Name': ('Name (GT)', 'Name (Pred)'), 'Application No.': ('Application No. (GT)', 'Application No. (Pred)'),
    'Test Date': ('Test Date (GT)', 'Test Date (Pred)'), 'Grammar_Level': ('Grammar_Level (GT)', 'Grammar_Level (Pred)'),
    'Grammar_Score': ('Grammar_Score (GT)', 'Grammar_Score (Pred)'), 'Reading_Level': ('Reading_Level (GT)', 'Reading_Level (Pred)'),
    'Reading_Score': ('Reading_Score (GT)', 'Reading_Score (Pred)'), 'Speaking_Level': ('Speaking_Level (GT)', 'Speaking_Level (Pred)'),
    'Speaking_Score': ('Speaking_Score (GT)', 'Speaking_Score (Pred)'), 'Writing_Level': ('Writing_Level (GT)', 'Writing_Level (Pred)'),
    'Writing_Score': ('Writing_Score (GT)', 'Writing_Score (Pred)'), 'Total_Level': ('Total_Level (GT)', 'Total_Level (Pred)'),
    'Total_Score': ('Total_Score (GT)', 'Total_Score (Pred)'),
}
evaluation_summary_list = []
for field, (gt_col, pred_col) in fields_to_evaluate.items():
    ground_truth = [normalize_text(t) for t in eval_df[gt_col]]
    prediction = [normalize_text(t) for t in eval_df[pred_col]]
    accuracy = np.mean([1 if gt == pred else 0 for gt, pred in zip(ground_truth, prediction)]) * 100
    try:
        error_metrics = jiwer.compute_measures(ground_truth, prediction)
        wer = error_metrics.get('wer', 0) * 100
        cer = error_metrics.get('cer', 0) * 100
        H = error_metrics.get('hits', 0); I = error_metrics.get('insertions', 0); D = error_metrics.get('deletions', 0); S = error_metrics.get('substitutions', 0)
        precision = H / (H + I + S) if (H + I + S) > 0 else 0
        recall = H / (H + D + S) if (H + D + S) > 0 else 0
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    except ValueError:
        wer, cer, f1_score = 100, 100, 0
    evaluation_summary_list.append({
        'Field': field, 'Accuracy (%)': round(accuracy, 2), 'WER (%)': round(wer, 2),
        'CER (%)': round(cer, 2), 'F1-score (%)': round(f1_score * 100, 2)
    })
eval_summary_df = pd.DataFrame(evaluation_summary_list)
eval_summary_df.insert(0, 'Approach', APPROACH_NAME)
eval_summary_df.insert(0, 'Model', MODEL_NAME)
print(eval_summary_df.to_string(index=False))
print("--- ✅ ประเมินผลเสร็จสิ้น ---\n")

# --- 6. บันทึกผลลัพธ์ ---
print(f"--- 💾 กำลังบันทึกรายงานเฉพาะของ {MODEL_NAME} ({APPROACH_NAME}) ---")
with pd.ExcelWriter(INDIVIDUAL_REPORT_PATH, engine='openpyxl') as writer:
    ocr_results_df.to_excel(writer, sheet_name='Detailed_Results', index=False)
    eval_summary_df.drop(columns=['Model', 'Approach']).to_excel(writer, sheet_name='Evaluation_Summary', index=False)
print(f"🎉 บันทึกไฟล์ {INDIVIDUAL_REPORT_PATH} สำเร็จ!")

print(f"--- 💾 กำลังอัปเดตไฟล์ Master Report: {MASTER_OUTPUT_PATH} ---")
SHEET_NAME = 'Master_Evaluation'
try:
    master_df = pd.read_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME)
    master_df = master_df[~((master_df['Model'] == MODEL_NAME) & (master_df['Approach'] == APPROACH_NAME))]
    combined_df = pd.concat([master_df, eval_summary_df], ignore_index=True)
except FileNotFoundError:
    print(f"ไม่พบไฟล์ Master เดิม, กำลังสร้างไฟล์ใหม่...")
    combined_df = eval_summary_df
combined_df.to_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME, index=False)
print(f"🎉 อัปเดตไฟล์ Master Report สำเร็จ!")

# EASY OCR

In [ ]:
import easyocr
import torch
from PIL import Image

# --- 1. กำหนดค่าและ Path ---
# ระบุ Path ของภาพที่ผ่าน Preprocess แล้ว (ไฟล์ขาวดำ)
IMAGE_PATH = "/content/drive/MyDrive/lastest-ktep/prep-img/KMITL-TEP PILOT-1.png"

# --- 2. โหลดโมเดล EasyOCR (ทำครั้งเดียว) ---
print("--- 🤖 กำลังโหลดโมเดล EasyOCR ---")
# ตรวจสอบว่ามี GPU ให้ใช้หรือไม่ เพื่อความรวดเร็ว
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# สร้าง Reader สำหรับภาษาอังกฤษ
easyocr_reader = easyocr.Reader(['en'], gpu=(DEVICE == 'cuda'))
print("--- ✅ โหลดโมเดลสำเร็จ! ---")


# --- 3. เริ่มกระบวนการสกัดข้อความดิบ ---
try:
    print(f"\n--- 🚀 กำลังอ่านข้อความจากรูปภาพ: {IMAGE_PATH} ---")

    # ใช้ EasyOCR อ่านทั้งหน้า
    # paragraph=True จะช่วยรวมกลุ่มข้อความที่อยู่ใกล้กันให้เป็นย่อหน้าเดียว
    easyocr_text_list = easyocr_reader.readtext(IMAGE_PATH, detail=0, paragraph=True)

    # นำผลลัพธ์ที่ได้ (ซึ่งเป็น list ของ string) มารวมกันโดยคั่นด้วยการขึ้นบรรทัดใหม่
    raw_text = "\n".join(easyocr_text_list)

    # --- 4. แสดงผลลัพธ์ ---
    print("\n" + "="*50)
    print("📄 RAW TEXT OUTPUT FROM EASYOCR (PURE OCR) 📄")
    print("="*50)
    # พิมพ์ผลลัพธ์ที่ได้ออกมาทั้งหมด
    print(raw_text)
    print("="*50)
    print("\n--- ✅ เสร็จสิ้น ---")
    print("โปรดตรวจสอบข้อความด้านบนเพื่อหา Pattern สำหรับขั้นตอนถัดไป")

except FileNotFoundError:
    print(f"\n[ERROR] ไม่พบไฟล์ที่ Path: {IMAGE_PATH}")
    print("กรุณาตรวจสอบว่า Path ของไฟล์ถูกต้องหรือไม่")
except Exception as e:
    print(f"\nAn error occurred: {e}")

In [ ]:
import pandas as pd
import re
import easyocr
import torch
from IPython.display import display

# --- 1. กำหนดค่าและ Path ---
PROCESSED_IMAGE_PATH = "/content/drive/MyDrive/lastest-ktep/prep-img/KMITL-TEP PILOT-1.png"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"
TARGET_ID = 1

# --- 2. โหลดโมเดล EasyOCR ---
print("Initializing EasyOCR model...")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
easyocr_reader = easyocr.Reader(['en'], gpu=(DEVICE == 'cuda'))
print("Model ready.")

# --- 3. Parser ฉบับสมบูรณ์สุดท้าย (v5) ---
def ultimate_final_parser(full_text):
    data = {}

    # ฟังก์ชันย่อยสำหรับทำความสะอาดและแยกส่วนคะแนน
    def parse_score_from_text(text):
        if not text: return None, None

        # Regex ที่ยืดหยุ่นสำหรับ Level(Score)
        match = re.search(r"([A-ZBIl1-9]+)\s*\(\s*(\d+)\)", text)
        if not match: return None, None

        level, score = match.group(1), match.group(2)

        # ทำความสะอาด Level
        cleaned_level = level.upper().replace('I', '1').replace('L', '1')
        if 'B11' in cleaned_level: cleaned_level = 'B1'

        return cleaned_level, score

    # --- สกัดข้อมูลส่วนหัว ---
    name_match = re.search(r"Name\s*:?\s*(.*?)\s*Application No", full_text, re.DOTALL | re.IGNORECASE)
    data['name'] = re.sub(r'\s+', ' ', name_match.group(1).strip()) if name_match else None

    # Pattern ที่ยืดหยุ่นขึ้นสำหรับ App No.
    app_no_match = re.search(r"Application\s*No\.?\s*:*\s*([A-Z0-9\-]+)", full_text, re.IGNORECASE)
    data['application_no'] = app_no_match.group(1).strip() if app_no_match else None

    date_match = re.search(r"Administration\s*:?\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", full_text, re.IGNORECASE)
    data['test_date'] = date_match.group(1).strip() if date_match else None

    # --- สกัดคะแนนด้วยวิธี Block-based ที่เสถียรที่สุด ---
    total_block = re.search(r"KMITL-TEP Level(.*?)(Grammar|$)", full_text, re.DOTALL | re.IGNORECASE)
    data['total_level'], data['total_score'] = parse_score_from_text(total_block.group(1) if total_block else "")

    grammar_block = re.search(r"Grammar(.*?)Reading", full_text, re.DOTALL | re.IGNORECASE)
    data['grammar_level'], data['grammar_score'] = parse_score_from_text(grammar_block.group(1) if grammar_block else "")

    reading_block = re.search(r"Reading(.*?)Speaking", full_text, re.DOTALL | re.IGNORECASE)
    data['reading_level'], data['reading_score'] = parse_score_from_text(reading_block.group(1) if reading_block else "")

    speaking_block = re.search(r"Speaking(.*?)Writing", full_text, re.DOTALL | re.IGNORECASE)
    data['speaking_level'], data['speaking_score'] = parse_score_from_text(speaking_block.group(1) if speaking_block else "")

    # ทำให้ Writing ทนทานขึ้นโดยหาจาก Writing ไปจนสุด หรือจนเจอคำว่า This is part
    writing_block = re.search(r"Writing(.*?)(This is part|$)", full_text, re.DOTALL | re.IGNORECASE)
    data['writing_level'], data['writing_score'] = parse_score_from_text(writing_block.group(1) if writing_block else "")

    return data

# --- 4. เริ่มกระบวนการ ---
try:
    # โหลด Ground Truth
    gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
    target_gt_row = gt_df[gt_df['No.'] == TARGET_ID].iloc[0]

    # สกัดข้อความดิบด้วย EasyOCR
    print("Running EasyOCR...")
    easyocr_text_list = easyocr_reader.readtext(PROCESSED_IMAGE_PATH, detail=0, paragraph=True)
    full_text = "\n".join(easyocr_text_list)

    # Parse ข้อความดิบด้วย Parser ที่ดีที่สุด
    print("Parsing extracted text...")
    extracted_data = ultimate_final_parser(full_text)

    # สร้างตารางเปรียบเทียบ
    comparison_data = []
    # ใช้ dict ของ ground truth ที่คุณให้มาเป็นตัวกำหนดโครงสร้าง
    gt_structure = {
        'Name': target_gt_row['Name'], 'Application No.': target_gt_row['Application No.'],
        'Test Date': target_gt_row['Test Date'], 'Grammar Level': target_gt_row['Grammar_Level'],
        'Grammar Score': target_gt_row['Grammar_Score'], 'Reading Level': target_gt_row['Reading_Level'],
        'Reading Score': target_gt_row['Reading_Score'], 'Speaking Level': target_gt_row['Speaking_Level'],
        'Speaking Score': target_gt_row['Speaking_Score'], 'Writing Level': target_gt_row['Writing_Level'],
        'Writing Score': target_gt_row['Writing_Score'], 'Total Level': target_gt_row['Total_Level'],
        'Total Score': target_gt_row['Total_Score']
    }

    pred_map = {
        'Name': 'name', 'Application No.': 'application_no', 'Test Date': 'test_date',
        'Grammar Level': 'grammar_level', 'Grammar Score': 'grammar_score',
        'Reading Level': 'reading_level', 'Reading Score': 'reading_score',
        'Speaking Level': 'speaking_level', 'Speaking Score': 'speaking_score',
        'Writing Level': 'writing_level', 'Writing Score': 'writing_score',
        'Total Level': 'total_level', 'Total Score': 'total_score'
    }

    for field, gt_value in gt_structure.items():
        pred_key = pred_map[field]
        comparison_data.append({
            "Field": field,
            "Ground Truth": gt_value,
            "Prediction": extracted_data.get(pred_key, "Not Found")
        })

    comparison_df = pd.DataFrame(comparison_data)

    print("\n--- ✅ Final Result: EasyOCR (Pure OCR) vs Ground Truth (Corrected) ---")
    display(comparison_df)

except Exception as e:
    print(f"\nAn error occurred: {e}")

In [ ]:
import cv2
import pandas as pd
import os
import re
import jiwer
import numpy as np
import easyocr  # <<< เพิ่ม import
import torch    # <<< เพิ่ม import
from tqdm import tqdm
from PIL import Image

# --- 1. กำหนดค่าและ Path ที่สำคัญ ---
MODEL_NAME = 'EasyOCR' # <<< เปลี่ยนชื่อโมเดล
APPROACH_NAME = 'Pure OCR'
MASTER_OUTPUT_PATH = "/content/drive/MyDrive/lastest-ktep/excel/comparison_report.xlsx"
INDIVIDUAL_REPORT_PATH = f"/content/drive/MyDrive/lastest-ktep/excel/{MODEL_NAME.lower()}_{APPROACH_NAME.replace(' ', '_').lower()}_report.xlsx"
IMAGE_DIR = "/content/drive/MyDrive/lastest-ktep/prep-img"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"

# --- 2. ฟังก์ชันช่วยเหลือ (ไม่มีการเปลี่ยนแปลง) ---
def get_id_from_filename(filename):
    match = re.search(r'(\d+)\.(png|jpg|jpeg)$', filename.lower())
    if match: return int(match.group(1))
    return None

def normalize_text(text):
    if text is None: return ""
    text_str = str(text).strip().lower().replace(' ', '')
    if text_str.endswith('.0'): text_str = text_str[:-2]
    return text_str

def final_robust_parser(full_text):
    data = {}
    def clean_level(level_text):
        if not level_text: return None
        cleaned = level_text.upper().replace('I', '1').replace('l', '1')
        if 'B11' in cleaned: return 'B1'
        return cleaned
    def parse_score(text):
        if not text: return None, None
        match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", text)
        return (clean_level(match.group(1)), match.group(2)) if match else (None, None)
    name_match = re.search(r"Name\s*:?\s*(.*?)\s*(?:Application No|Date of test)", full_text, re.DOTALL | re.IGNORECASE)
    data['name'] = re.sub(r'\s+', ' ', name_match.group(1).strip()) if name_match else None
    app_no_match = re.search(r"Application No\.?\s*:*\s*([A-Z0-9\-]+)", full_text, re.IGNORECASE)
    data['application_no'] = app_no_match.group(1).strip() if app_no_match else None
    date_match = re.search(r"Administration\s*:?\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", full_text, re.IGNORECASE)
    data['test_date'] = date_match.group(1).strip() if date_match else None
    total_block = re.search(r"KMITL-TEP Level(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['total_level'], data['total_score'] = parse_score(total_block.group(1) if total_block else "")
    grammar_block = re.search(r"Grammar(.*?)Reading", full_text, re.DOTALL | re.IGNORECASE)
    data['grammar_level'], data['grammar_score'] = parse_score(grammar_block.group(1) if grammar_block else "")
    reading_block = re.search(r"Reading(.*?)Speaking", full_text, re.DOTALL | re.IGNORECASE)
    data['reading_level'], data['reading_score'] = parse_score(reading_block.group(1) if reading_block else "")
    speaking_block = re.search(r"Speaking(.*?)Writing", full_text, re.DOTALL | re.IGNORECASE)
    data['speaking_level'], data['speaking_score'] = parse_score(speaking_block.group(1) if speaking_block else "")
    writing_block = re.search(r"Writing(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['writing_level'], data['writing_score'] = parse_score(writing_block.group(1) if writing_block else "")
    return data

# --- 3. โหลดโมเดลและข้อมูล ---
print("--- 🔍 ขั้นตอนโหลดโมเดลและข้อมูล ---")
# ⬇️⬇️⬇️ โหลดโมเดล EasyOCR ⬇️⬇️⬇️
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
easyocr_reader = easyocr.Reader(['en'], gpu=(DEVICE == 'cuda'))
print(f"EasyOCR model loaded, using device: {DEVICE}")
# ⬆️⬆️⬆️ สิ้นสุดการโหลดโมเดล ⬆️⬆️⬆️

gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
gt_df['No.'] = gt_df['No.'].astype(int)
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))], key=lambda f: get_id_from_filename(f) or 0)
print(f"พบ ID ใน Ground Truth ทั้งหมด: {len(gt_df['No.'].unique())} ID | พบไฟล์รูปภาพ: {len(image_files)} ไฟล์")
print("--- ✅ ตรวจสอบข้อมูลเสร็จสิ้น ---\n")

# --- 4. เริ่มกระบวนการสกัดข้อมูล ---
results_list = []
print(f"--- 🚀 เริ่มการสกัดข้อมูลจากรูปภาพ {len(image_files)} ไฟล์ ด้วย {MODEL_NAME} ({APPROACH_NAME}) ---")
for filename in tqdm(image_files, desc=f"Processing with {MODEL_NAME}"):
    image_path = os.path.join(IMAGE_DIR, filename)
    image_id = get_id_from_filename(filename)
    if image_id is None: continue

    gt_row_df = gt_df[gt_df['No.'] == image_id]
    if gt_row_df.empty: continue
    gt_row = gt_row_df.iloc[0]

    try:
        # ⬇️⬇️⬇️ เปลี่ยนมาใช้ EasyOCR ในการสกัดข้อความ ⬇️⬇️⬇️
        easyocr_text_list = easyocr_reader.readtext(image_path, detail=0, paragraph=True)
        full_text = "\n".join(easyocr_text_list)
        # ⬆️⬆️⬆️ สิ้นสุดส่วนที่เปลี่ยน ⬆️⬆️⬆️

        extracted_data = final_robust_parser(full_text)
    except Exception as ocr_error:
        print(f"Error processing file {filename}: {ocr_error}")
        extracted_data = {}

    results_list.append({
        "No.": image_id,
        "Application No. (GT)": gt_row["Application No."], "Application No. (Pred)": extracted_data.get('application_no'),
        "Name (GT)": gt_row["Name"], "Name (Pred)": extracted_data.get('name'),
        "Test Date (GT)": gt_row["Test Date"], "Test Date (Pred)": extracted_data.get('test_date'),
        "Grammar_Level (GT)": gt_row["Grammar_Level"], "Grammar_Level (Pred)": extracted_data.get('grammar_level'),
        "Grammar_Score (GT)": gt_row["Grammar_Score"], "Grammar_Score (Pred)": extracted_data.get('grammar_score'),
        "Reading_Level (GT)": gt_row["Reading_Level"], "Reading_Level (Pred)": extracted_data.get('reading_level'),
        "Reading_Score (GT)": gt_row["Reading_Score"], "Reading_Score (Pred)": extracted_data.get('reading_score'),
        "Speaking_Level (GT)": gt_row["Speaking_Level"], "Speaking_Level (Pred)": extracted_data.get('speaking_level'),
        "Speaking_Score (GT)": gt_row["Speaking_Score"], "Speaking_Score (Pred)": extracted_data.get('speaking_score'),
        "Writing_Level (GT)": gt_row["Writing_Level"], "Writing_Level (Pred)": extracted_data.get('writing_level'),
        "Writing_Score (GT)": gt_row["Writing_Score"], "Writing_Score (Pred)": extracted_data.get('writing_score'),
        "Total_Level (GT)": gt_row["Total_Level"], "Total_Level (Pred)": extracted_data.get('total_level'),
        "Total_Score (GT)": gt_row["Total_Score"], "Total_Score (Pred)": extracted_data.get('total_score'),
    })

ocr_results_df = pd.DataFrame(results_list)
print("--- ✅ สกัดข้อมูลเสร็จสิ้น ---\n")

# --- 5. เริ่มกระบวนการประเมินผล (Evaluation) ---
print("--- 📊 เริ่มการประเมินผล ---")
eval_df = ocr_results_df.fillna('')
fields_to_evaluate = {
    'Name': ('Name (GT)', 'Name (Pred)'), 'Application No.': ('Application No. (GT)', 'Application No. (Pred)'),
    'Test Date': ('Test Date (GT)', 'Test Date (Pred)'), 'Grammar_Level': ('Grammar_Level (GT)', 'Grammar_Level (Pred)'),
    'Grammar_Score': ('Grammar_Score (GT)', 'Grammar_Score (Pred)'), 'Reading_Level': ('Reading_Level (GT)', 'Reading_Level (Pred)'),
    'Reading_Score': ('Reading_Score (GT)', 'Reading_Score (Pred)'), 'Speaking_Level': ('Speaking_Level (GT)', 'Speaking_Level (Pred)'),
    'Speaking_Score': ('Speaking_Score (GT)', 'Speaking_Score (Pred)'), 'Writing_Level': ('Writing_Level (GT)', 'Writing_Level (Pred)'),
    'Writing_Score': ('Writing_Score (GT)', 'Writing_Score (Pred)'), 'Total_Level': ('Total_Level (GT)', 'Total_Level (Pred)'),
    'Total_Score': ('Total_Score (GT)', 'Total_Score (Pred)'),
}
evaluation_summary_list = []
for field, (gt_col, pred_col) in fields_to_evaluate.items():
    ground_truth = [normalize_text(t) for t in eval_df[gt_col]]
    prediction = [normalize_text(t) for t in eval_df[pred_col]]
    accuracy = np.mean([1 if gt == pred else 0 for gt, pred in zip(ground_truth, prediction)]) * 100
    try:
        error_metrics = jiwer.compute_measures(ground_truth, prediction)
        wer = error_metrics.get('wer', 0) * 100
        cer = error_metrics.get('cer', 0) * 100
        H = error_metrics.get('hits', 0); I = error_metrics.get('insertions', 0); D = error_metrics.get('deletions', 0); S = error_metrics.get('substitutions', 0)
        precision = H / (H + I + S) if (H + I + S) > 0 else 0
        recall = H / (H + D + S) if (H + D + S) > 0 else 0
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    except ValueError:
        wer, cer, f1_score = 100, 100, 0
    evaluation_summary_list.append({
        'Field': field, 'Accuracy (%)': round(accuracy, 2), 'WER (%)': round(wer, 2),
        'CER (%)': round(cer, 2), 'F1-score (%)': round(f1_score * 100, 2)
    })
eval_summary_df = pd.DataFrame(evaluation_summary_list)
eval_summary_df.insert(0, 'Approach', APPROACH_NAME)
eval_summary_df.insert(0, 'Model', MODEL_NAME)
print(eval_summary_df.to_string(index=False))
print("--- ✅ ประเมินผลเสร็จสิ้น ---\n")

# --- 6. บันทึกผลลัพธ์ ---
print(f"--- 💾 กำลังบันทึกรายงานเฉพาะของ {MODEL_NAME} ({APPROACH_NAME}) ---")
with pd.ExcelWriter(INDIVIDUAL_REPORT_PATH, engine='openpyxl') as writer:
    ocr_results_df.to_excel(writer, sheet_name='Detailed_Results', index=False)
    eval_summary_df.drop(columns=['Model', 'Approach']).to_excel(writer, sheet_name='Evaluation_Summary', index=False)
print(f"🎉 บันทึกไฟล์ {INDIVIDUAL_REPORT_PATH} สำเร็จ!")

print(f"--- 💾 กำลังอัปเดตไฟล์ Master Report: {MASTER_OUTPUT_PATH} ---")
SHEET_NAME = 'Master_Evaluation'
try:
    master_df = pd.read_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME)
    master_df = master_df[~((master_df['Model'] == MODEL_NAME) & (master_df['Approach'] == APPROACH_NAME))]
    combined_df = pd.concat([master_df, eval_summary_df], ignore_index=True)
except FileNotFoundError:
    print(f"ไม่พบไฟล์ Master เดิม, กำลังสร้างไฟล์ใหม่...")
    combined_df = eval_summary_df
combined_df.to_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME, index=False)
print(f"🎉 อัปเดตไฟล์ Master Report สำเร็จ!")

**Boxing**

In [ ]:
import cv2
import pandas as pd
import os
import re
import jiwer
import numpy as np
from tqdm import tqdm
from PIL import Image
import easyocr
import torch

# --- 1. กำหนดค่าและ Path ---
MODEL_NAME = 'EasyOCR'
APPROACH_NAME = 'Boxing OCR'
MASTER_OUTPUT_PATH = "/content/drive/MyDrive/lastest-ktep/excel/comparison_report.xlsx"
INDIVIDUAL_REPORT_PATH = f"/content/drive/MyDrive/lastest-ktep/excel/{MODEL_NAME.lower()}_{APPROACH_NAME.replace(' ', '_').lower()}_report.xlsx"
IMAGE_DIR = "/content/drive/MyDrive/lastest-ktep/prep-img"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"

ROI_CONFIG = {
    'name': (200, 485, 600, 65), 'application_no': (1100, 485, 300, 60),
    'test_date': (1310, 545, 300, 60), 'overall_score': (820, 625, 800, 100),
    'grammar': (320, 785, 500, 80), 'reading': (1050, 785, 500, 80),
    'speaking': (320, 900, 500, 80), 'writing': (1050, 900, 500, 80),
}

# --- 2. ฟังก์ชันช่วยเหลือ ---
def get_id_from_filename(filename):
    match = re.search(r'(\d+)\.(png|jpg|jpeg)$', filename.lower())
    if match: return int(match.group(1))
    return None

def normalize_text(text):
    if text is None: return ""
    text_str = str(text).strip().lower().replace(' ', '')
    if text_str.endswith('.0'): text_str = text_str[:-2]
    return text_str

# ⬇️⬇️⬇️ ฟังก์ชัน Parse ฉบับแก้ไขสมบูรณ์ ⬇️⬇️⬇️
def parse_level_and_score(text):
    if not text: return None, None
    cleaned_text = text.replace('l', '1').replace('I', '1')
    cleaned_text = cleaned_text.upper()
    match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", cleaned_text)
    if match:
        level, score = match.group(1), match.group(2)
        if len(level) > 2 and level.startswith('B'): level = 'B1'
        return level, score
    return None, None
# ⬆️⬆️⬆️ สิ้นสุดฟังก์ชันที่แก้ไข ⬆️⬆️⬆️

# --- 3. โหลดโมเดลและข้อมูล ---
print("--- 🔍 ขั้นตอนโหลดโมเดลและข้อมูล ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
easyocr_reader = easyocr.Reader(['en'], gpu=(DEVICE == 'cuda'))
print(f"EasyOCR model loaded, using device: {DEVICE}")

gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
gt_df['No.'] = gt_df['No.'].astype(int)
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))], key=lambda f: get_id_from_filename(f) or 0)
print(f"พบ ID ใน Ground Truth ทั้งหมด: {len(gt_df['No.'].unique())} ID | พบไฟล์รูปภาพ: {len(image_files)} ไฟล์")
print("--- ✅ ตรวจสอบข้อมูลเสร็จสิ้น ---\n")

# --- 4. เริ่มกระบวนการสกัดข้อมูล ---
results_list = []
print(f"--- 🚀 เริ่มการสกัดข้อมูลจากรูปภาพ {len(image_files)} ไฟล์ ด้วย {MODEL_NAME} ({APPROACH_NAME}) ---")
for filename in tqdm(image_files, desc=f"Processing with {MODEL_NAME}"):
    image_id = get_id_from_filename(filename)
    if image_id is None: continue
    gt_row_df = gt_df[gt_df['No.'] == image_id]
    if gt_row_df.empty: continue
    gt_row = gt_row_df.iloc[0]
    image_path = os.path.join(IMAGE_DIR, filename)
    try:
        image_pil = Image.open(image_path)
        extracted_data = {}
        for field, (x, y, w, h) in ROI_CONFIG.items():
            x1, y1, x2, y2 = x, y, x + w, y + h
            cropped_box = image_pil.crop((x1, y1, x2, y2))
            result = easyocr_reader.readtext(np.array(cropped_box), detail=0, paragraph=False)
            text = " ".join(result).strip()
            if field in ['grammar', 'reading', 'speaking', 'writing', 'overall_score']:
                level, score = parse_level_and_score(text)
                if field == 'overall_score':
                    extracted_data['total_level'], extracted_data['total_score'] = level, score
                else:
                    extracted_data[f'{field}_level'], extracted_data[f'{field}_score'] = level, score
            else:
                extracted_data[field] = text
    except Exception as ocr_error:
        print(f"Error processing file {filename}: {ocr_error}")
        extracted_data = {}
    results_list.append({
        "No.": image_id,
        "Application No. (GT)": gt_row["Application No."], "Application No. (Pred)": extracted_data.get('application_no'),
        "Name (GT)": gt_row["Name"], "Name (Pred)": extracted_data.get('name'),
        "Test Date (GT)": gt_row["Test Date"], "Test Date (Pred)": extracted_data.get('test_date'),
        "Grammar_Level (GT)": gt_row["Grammar_Level"], "Grammar_Level (Pred)": extracted_data.get('grammar_level'),
        "Grammar_Score (GT)": gt_row["Grammar_Score"], "Grammar_Score (Pred)": extracted_data.get('grammar_score'),
        "Reading_Level (GT)": gt_row["Reading_Level"], "Reading_Level (Pred)": extracted_data.get('reading_level'),
        "Reading_Score (GT)": gt_row["Reading_Score"], "Reading_Score (Pred)": extracted_data.get('reading_score'),
        "Speaking_Level (GT)": gt_row["Speaking_Level"], "Speaking_Level (Pred)": extracted_data.get('speaking_level'),
        "Speaking_Score (GT)": gt_row["Speaking_Score"], "Speaking_Score (Pred)": extracted_data.get('speaking_score'),
        "Writing_Level (GT)": gt_row["Writing_Level"], "Writing_Level (Pred)": extracted_data.get('writing_level'),
        "Writing_Score (GT)": gt_row["Writing_Score"], "Writing_Score (Pred)": extracted_data.get('writing_score'),
        "Total_Level (GT)": gt_row["Total_Level"], "Total_Level (Pred)": extracted_data.get('total_level'),
        "Total_Score (GT)": gt_row["Total_Score"], "Total_Score (Pred)": extracted_data.get('total_score'),
    })
ocr_results_df = pd.DataFrame(results_list)
print("--- ✅ สกัดข้อมูลเสร็จสิ้น ---\n")

# --- 5 & 6. การประเมินผลและบันทึก ---
# --- 5. เริ่มกระบวนการประเมินผล (Evaluation) ---
print("--- 📊 เริ่มการประเมินผล ---")
eval_df = ocr_results_df.fillna('')
fields_to_evaluate = {
    'Name': ('Name (GT)', 'Name (Pred)'), 'Application No.': ('Application No. (GT)', 'Application No. (Pred)'),
    'Test Date': ('Test Date (GT)', 'Test Date (Pred)'), 'Grammar_Level': ('Grammar_Level (GT)', 'Grammar_Level (Pred)'),
    'Grammar_Score': ('Grammar_Score (GT)', 'Grammar_Score (Pred)'), 'Reading_Level': ('Reading_Level (GT)', 'Reading_Level (Pred)'),
    'Reading_Score': ('Reading_Score (GT)', 'Reading_Score (Pred)'), 'Speaking_Level': ('Speaking_Level (GT)', 'Speaking_Level (Pred)'),
    'Speaking_Score': ('Speaking_Score (GT)', 'Speaking_Score (Pred)'), 'Writing_Level': ('Writing_Level (GT)', 'Writing_Level (Pred)'),
    'Writing_Score': ('Writing_Score (GT)', 'Writing_Score (Pred)'), 'Total_Level': ('Total_Level (GT)', 'Total_Level (Pred)'),
    'Total_Score': ('Total_Score (GT)', 'Total_Score (Pred)'),
}
evaluation_summary_list = []
for field, (gt_col, pred_col) in fields_to_evaluate.items():
    ground_truth = [normalize_text(t) for t in eval_df[gt_col]]
    prediction = [normalize_text(t) for t in eval_df[pred_col]]
    accuracy = np.mean([1 if gt == pred else 0 for gt, pred in zip(ground_truth, prediction)]) * 100
    try:
        error_metrics = jiwer.compute_measures(ground_truth, prediction)
        wer = error_metrics.get('wer', 0) * 100
        cer = error_metrics.get('cer', 0) * 100
        H = error_metrics.get('hits', 0); I = error_metrics.get('insertions', 0); D = error_metrics.get('deletions', 0); S = error_metrics.get('substitutions', 0)
        precision = H / (H + I + S) if (H + I + S) > 0 else 0
        recall = H / (H + D + S) if (H + D + S) > 0 else 0
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    except ValueError:
        wer, cer, f1_score = 100, 100, 0
    evaluation_summary_list.append({
        'Field': field, 'Accuracy (%)': round(accuracy, 2), 'WER (%)': round(wer, 2),
        'CER (%)': round(cer, 2), 'F1-score (%)': round(f1_score * 100, 2)
    })
eval_summary_df = pd.DataFrame(evaluation_summary_list)
eval_summary_df.insert(0, 'Approach', APPROACH_NAME)
eval_summary_df.insert(0, 'Model', MODEL_NAME)
print(eval_summary_df.to_string(index=False))
print("--- ✅ ประเมินผลเสร็จสิ้น ---\n")

# --- 6. บันทึกผลลัพธ์ ---
print(f"--- 💾 กำลังบันทึกรายงานเฉพาะของ {MODEL_NAME} ({APPROACH_NAME}) ---")
with pd.ExcelWriter(INDIVIDUAL_REPORT_PATH, engine='openpyxl') as writer:
    ocr_results_df.to_excel(writer, sheet_name='Detailed_Results', index=False)
    eval_summary_df.drop(columns=['Model', 'Approach']).to_excel(writer, sheet_name='Evaluation_Summary', index=False)
print(f"🎉 บันทึกไฟล์ {INDIVIDUAL_REPORT_PATH} สำเร็จ!")

print(f"--- 💾 กำลังอัปเดตไฟล์ Master Report: {MASTER_OUTPUT_PATH} ---")
SHEET_NAME = 'Master_Evaluation'
try:
    master_df = pd.read_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME)
    master_df = master_df[~((master_df['Model'] == MODEL_NAME) & (master_df['Approach'] == APPROACH_NAME))]
    combined_df = pd.concat([master_df, eval_summary_df], ignore_index=True)
except FileNotFoundError:
    print(f"ไม่พบไฟล์ Master เดิม, กำลังสร้างไฟล์ใหม่...")
    combined_df = eval_summary_df
combined_df.to_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME, index=False)
print(f"🎉 อัปเดตไฟล์ Master Report สำเร็จ!")

# Keras OCR ทำไม่ได้วุ้ยยย

คำอธิบาย: เกิดอะไรขึ้น?
สถานการณ์ตอนนี้เปรียบเสมือนเรามีอุปกรณ์ 2 ชิ้นที่ใช้ปลั๊กไฟคนละแบบและไม่สามารถเสียบพร้อมกันได้ครับ:

Keras-OCR (ที่เราอยากใช้): เป็นไลบรารีที่ค่อนข้างเก่า และถูกสร้างมาให้ทำงานกับ NumPy เวอร์ชัน 1.x
Thinc (ไลบรารีอื่นที่ติดตั้งอยู่ใน Colab อยู่แล้ว): เป็นไลบรารีที่ใหม่กว่า และมันต้องการ NumPy เวอร์ชัน 2.x
เมื่อเราใช้คำสั่ง !pip install "numpy<2.0" เพื่อ Downgrade ให้ Keras-OCR ทำงานได้ มันก็ไปทำให้ไลบรารี Thinc พังลงทันทีเพราะเวอร์ชันไม่เข้ากัน ซึ่งปัญหานี้แก้ไขได้ยากมากและอยู่นอกเหนือขอบเขตของโปรเจกต์ OCR ของเราครับ

คำแนะนำในฐานะผู้เชี่ยวชาญ
ในฐานะผู้เชี่ยวชาญ ผมขอแนะนำว่าเราควร ยุติการทดลองกับ Keras-OCR ไว้เพียงเท่านี้ครับ

เนื่องจากตัวไลบรารี keras-ocr เองไม่ได้ถูกอัปเดตให้เข้ากับสภาพแวดล้อมใหม่ๆ (เช่น NumPy 2.0) มาเป็นเวลานานแล้ว การพยายามฝืนใช้งานต่อจะสร้างปัญหาทางเทคนิคที่ซับซ้อนมากกว่าประโยชน์ที่เราจะได้รับจากการเปรียบเทียบครับ

In [ ]:
# --- เซลล์สำหรับติดตั้ง Keras-OCR และจัดการ Dependency Conflict ---

# 1. ถอนการติดตั้งไลบรารีที่ขัดแย้งกันออกไปก่อน
!pip uninstall -y thinc spacy

# 2. บังคับติดตั้ง NumPy เวอร์ชันที่เข้ากันได้ (เวอร์ชันเก่ากว่า 2.0)
!pip install "numpy<2.0" -q

# 3. ติดตั้ง Keras-OCR
!pip install keras-ocr -q

print("\n✅ ติดตั้งสำเร็จ! กรุณาทำขั้นตอนต่อไป")

In [ ]:
# --- โค้ดสำหรับทดสอบ Keras-OCR (Pure OCR) - Single Image ---
import pandas as pd
import re
import keras_ocr
from IPython.display import display

# --- 1. กำหนดค่าและ Path ---
IMAGE_PATH = "/content/drive/MyDrive/lastest-ktep/prep-img/1.png"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"
TARGET_ID = 1

# --- 2. Parser ที่ดีที่สุดของเรา ---
def final_robust_parser(full_text):
    data = {}
    def clean_level(level_text):
        if not level_text: return None
        cleaned = level_text.upper().replace('I', '1').replace('L', '1')
        if 'B11' in cleaned: return 'B1'
        return cleaned
    def parse_score(text):
        if not text: return None, None
        match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", text)
        return (clean_level(match.group(1)), match.group(2)) if match else (None, None)

    name_match = re.search(r"Name\s*:?\s*(.*?)\s*(?:Application No|Date of test)", full_text, re.DOTALL | re.IGNORECASE)
    data['name'] = re.sub(r'\s+', ' ', name_match.group(1).strip()) if name_match else None
    app_no_match = re.search(r"Application\s*No\.?\s*:*\s*([A-Z0-9\-]+)", full_text, re.IGNORECASE)
    data['application_no'] = app_no_match.group(1).strip() if app_no_match else None
    date_match = re.search(r"Administration\s*:?\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", full_text, re.IGNORECASE)
    data['test_date'] = date_match.group(1).strip() if date_match else None
    total_block = re.search(r"KMITL-TEP Level(.*?)(Grammar|$)", full_text, re.DOTALL | re.IGNORECASE)
    data['total_level'], data['total_score'] = parse_score(total_block.group(1) if total_block else "")
    grammar_block = re.search(r"Grammar(.*?)Reading", full_text, re.DOTALL | re.IGNORECASE)
    data['grammar_level'], data['grammar_score'] = parse_score(grammar_block.group(1) if grammar_block else "")
    reading_block = re.search(r"Reading(.*?)Speaking", full_text, re.DOTALL | re.IGNORECASE)
    data['reading_level'], data['reading_score'] = parse_score(reading_block.group(1) if reading_block else "")
    speaking_block = re.search(r"Speaking(.*?)Writing", full_text, re.DOTALL | re.IGNORECASE)
    data['speaking_level'], data['speaking_score'] = parse_score(speaking_block.group(1) if speaking_block else "")
    writing_block = re.search(r"Writing(.*?)(This is part|$)", full_text, re.DOTALL | re.IGNORECASE)
    data['writing_level'], data['writing_score'] = parse_score(writing_block.group(1) if writing_block else "")
    return data

# --- 3. เริ่มกระบวนการทั้งหมด ---
try:
    # 3.1 โหลดโมเดล Keras-OCR
    print("--- 🤖 กำลังโหลดโมเดล Keras-OCR ---")
    pipeline = keras_ocr.pipeline.Pipeline()
    print("--- ✅ โหลดโมเดลสำเร็จ! ---")

    # 3.2 โหลด Ground Truth
    gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
    target_gt_row = gt_df[gt_df['No.'] == TARGET_ID].iloc[0]

    # 3.3 สกัดข้อความดิบด้วย Keras-OCR
    print(f"--- 🚀 กำลังประมวลผลรูปภาพ ID: {TARGET_ID} ด้วย Keras-OCR ---")
    image = keras_ocr.tools.read(IMAGE_PATH)
    prediction_groups = pipeline.recognize([image])

    # Keras-OCR คืนค่าพิกัดมาด้วย เราสามารถจัดเรียงบรรทัดเพื่อสร้าง full_text ที่ดีขึ้นได้
    lines = {}
    for word, box in prediction_groups[0]:
        # จัดกลุ่มคำโดยใช้ค่า y เฉลี่ยของ bounding box เป็นตัวระบุบรรทัด
        line_y = np.mean([point[1] for point in box])
        # ปัดค่า y เพื่อให้คำในบรรทัดเดียวกันถูกจัดกลุ่มง่ายขึ้น
        line_key = round(line_y / 20) # 20 คือค่าประมาณความสูงของบรรทัด
        if line_key not in lines:
            lines[line_key] = []
        lines[line_key].append((np.mean([point[0] for point in box]), word))

    # เรียงลำดับคำในแต่ละบรรทัดจากซ้ายไปขวา แล้วรวมทุกบรรทัด
    full_text_list = []
    for line_key in sorted(lines.keys()):
        line_words = [word for x, word in sorted(lines[line_key])]
        full_text_list.append(" ".join(line_words))

    full_text = "\n".join(full_text_list)

    # 3.4 Parse ข้อความดิบ
    print("--- 🧠 กำลังสกัดข้อมูลด้วย Parser ---")
    extracted_data = final_robust_parser(full_text)

    # 3.5 สร้างตารางเปรียบเทียบ
    comparison_data = []
    gt_structure = {
        'Name': target_gt_row['Name'], 'Application No.': target_gt_row['Application No.'],
        'Test Date': target_gt_row['Test Date'], 'Grammar Level': target_gt_row['Grammar_Level'],
        'Grammar Score': target_gt_row['Grammar_Score'], 'Reading Level': target_gt_row['Reading_Level'],
        'Reading Score': target_gt_row['Reading_Score'], 'Speaking Level': target_gt_row['Speaking_Level'],
        'Speaking Score': target_gt_row['Speaking_Score'], 'Writing Level': target_gt_row['Writing_Level'],
        'Writing Score': target_gt_row['Writing_Score'], 'Total Level': target_gt_row['Total_Level'],
        'Total Score': target_gt_row['Total_Score']
    }
    pred_map = {
        'Name': 'name', 'Application No.': 'application_no', 'Test Date': 'test_date',
        'Grammar Level': 'grammar_level', 'Grammar Score': 'grammar_score',
        'Reading Level': 'reading_level', 'Reading Score': 'reading_score',
        'Speaking Level': 'speaking_level', 'Speaking Score': 'speaking_score',
        'Writing Level': 'writing_level', 'Writing Score': 'writing_score',
        'Total Level': 'total_level', 'Total Score': 'total_score'
    }
    for field, gt_value in gt_structure.items():
        pred_key = pred_map[field]
        comparison_data.append({
            "Field": field, "Ground Truth": gt_value,
            "Prediction (Keras-OCR)": extracted_data.get(pred_key, "Not Found")
        })
    comparison_df = pd.DataFrame(comparison_data)

    # 3.6 แสดงผลลัพธ์
    print("\n--- ✅ Final Result: Keras-OCR (Pure OCR) vs Ground Truth ---")
    display(comparison_df)

except IndexError:
    print(f"\n[ERROR] ไม่พบ ID: {TARGET_ID} ในไฟล์ Excel '{GROUND_TRUTH_PATH}'")
except Exception as e:
    print(f"\nAn error occurred: {e}")

# TR OCR

1. TrOCR ไม่ได้ถูกออกแบบมาสำหรับ Pure OCR: สถาปัตยกรรมของ TrOCR ถูกสร้างมาเพื่ออ่านข้อความที่ถูกตัดมาแล้วเป็น "หนึ่งบรรทัด" (Single Line) ได้อย่างแม่นยำสูงสุด
2. ปัญหาการอ่านทั้งหน้า: การนำภาพเอกสารทั้งหน้าที่มี Layout ซับซ้อนไปให้ TrOCR อ่านโดยตรง จะทำให้โมเดลสับสนและให้ผลลัพธ์ที่ผิดพลาดสูงมาก (เหมือนที่เราเคยเจอตัวอักษรขยะ)
3. วิธีแก้ปัญหาเฉพาะหน้า (Workaround): เพื่อให้การทดลองนี้ "เกิดขึ้นได้" สคริปต์ที่ผมจะให้ไปจึงต้องใช้วิธีแบบผสม คือ ใช้ EasyOCR มาช่วยหาตำแหน่งและตัดแบ่งบรรทัดให้ก่อน แล้วจึงส่งแต่ละบรรทัดที่ตัดแล้วไปให้ TrOCR อ่าน ซึ่งกระบวนการนี้จะทำงานช้ากว่าแนวทางอื่นๆ มาก

In [ ]:
import cv2
import pandas as pd
import os
import re
import jiwer
import numpy as np
from tqdm import tqdm
from PIL import Image
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import pytesseract

# --- 1. กำหนดค่าและ Path ที่สำคัญ ---
MODEL_NAME = 'TrOCR'
APPROACH_NAME = 'Pure OCR (Hybrid)' # <<< ระบุว่าเป็นแนวทางไฮบริด
MASTER_OUTPUT_PATH = "/content/drive/MyDrive/lastest-ktep/excel/comparison_report.xlsx"
INDIVIDUAL_REPORT_PATH = f"/content/drive/MyDrive/lastest-ktep/excel/{MODEL_NAME.lower()}_{APPROACH_NAME.replace(' ', '_').lower()}_report.xlsx"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"

# ⬇️⬇️⬇️ สำคัญ: ให้ใช้โฟลเดอร์ที่เก็บ "รูปสีต้นฉบับ" ⬇️⬇️⬇️
IMAGE_DIR = "/content/drive/MyDrive/lastest-ktep/prep-img" # <--- แก้ไข Path ให้เป็นโฟลเดอร์รูปสี

# --- 2. ฟังก์ชันช่วยเหลือ ---
def get_id_from_filename(filename):
    match = re.search(r'(\d+)\.(png|jpg|jpeg)$', filename.lower())
    if match: return int(match.group(1))
    return None

def normalize_text(text):
    if text is None: return ""
    text_str = str(text).strip().lower().replace(' ', '')
    if text_str.endswith('.0'): text_str = text_str[:-2]
    return text_str

# ⬇️⬇️⬇️ ใช้ Parser ที่ดีที่สุดของเรา ⬇️⬇️⬇️
def final_robust_parser(full_text):
    data = {}
    def clean_level(level_text):
        if not level_text: return None
        cleaned = level_text.upper().replace('I', '1').replace('l', '1')
        if 'B11' in cleaned: return 'B1'
        return cleaned
    def parse_score(text):
        if not text: return None, None
        match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", text)
        return (clean_level(match.group(1)), match.group(2)) if match else (None, None)
    name_match = re.search(r"Name\s*:?\s*(.*?)\s*(?:Application No|Date of test)", full_text, re.DOTALL | re.IGNORECASE)
    data['name'] = re.sub(r'\s+', ' ', name_match.group(1).strip()) if name_match else None
    app_no_match = re.search(r"Application No\.?\s*:*\s*([A-Z0-9\-]+)", full_text, re.IGNORECASE)
    data['application_no'] = app_no_match.group(1).strip() if app_no_match else None
    date_match = re.search(r"Administration\s*:?\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", full_text, re.IGNORECASE)
    data['test_date'] = date_match.group(1).strip() if date_match else None
    total_block = re.search(r"KMITL-TEP Level(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['total_level'], data['total_score'] = parse_score(total_block.group(1) if total_block else "")
    grammar_block = re.search(r"Grammar(.*?)Reading", full_text, re.DOTALL | re.IGNORECASE)
    data['grammar_level'], data['grammar_score'] = parse_score(grammar_block.group(1) if grammar_block else "")
    reading_block = re.search(r"Reading(.*?)Speaking", full_text, re.DOTALL | re.IGNORECASE)
    data['reading_level'], data['reading_score'] = parse_score(reading_block.group(1) if reading_block else "")
    speaking_block = re.search(r"Speaking(.*?)Writing", full_text, re.DOTALL | re.IGNORECASE)
    data['speaking_level'], data['speaking_score'] = parse_score(speaking_block.group(1) if speaking_block else "")
    writing_block = re.search(r"Writing(.*)", full_text, re.DOTALL | re.IGNORECASE)
    data['writing_level'], data['writing_score'] = parse_score(writing_block.group(1) if writing_block else "")
    return data

# --- 3. โหลดโมเดลและข้อมูล ---
print("--- 🔍 ขั้นตอนโหลดโมเดลและข้อมูล ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed').to(DEVICE)
print(f"TrOCR model loaded, using device: {DEVICE}")

gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
gt_df['No.'] = gt_df['No.'].astype(int)
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))], key=lambda f: get_id_from_filename(f) or 0)
print(f"พบ ID ใน Ground Truth ทั้งหมด: {len(gt_df['No.'].unique())} ID | พบไฟล์รูปภาพ: {len(image_files)} ไฟล์")
print("--- ✅ ตรวจสอบข้อมูลเสร็จสิ้น ---\n")

# --- 4. เริ่มกระบวนการสกัดข้อมูล ---
results_list = []
print(f"--- 🚀 เริ่มการสกัดข้อมูลจากรูปภาพ {len(image_files)} ไฟล์ ด้วย {MODEL_NAME} ({APPROACH_NAME}) ---")
for filename in tqdm(image_files, desc=f"Processing with {MODEL_NAME}"):
    image_id = get_id_from_filename(filename)
    if image_id is None: continue

    gt_row_df = gt_df[gt_df['No.'] == image_id]
    if gt_row_df.empty: continue
    gt_row = gt_row_df.iloc[0]

    image_path = os.path.join(IMAGE_DIR, filename)

    try:
        # --- ส่วนของ Hybrid Approach ---
        # 1. ใช้ Tesseract หาตำแหน่งของทุกคำในภาพสี
        line_data = pytesseract.image_to_data(image_path, config=r'--oem 3 --psm 3', output_type=pytesseract.Output.DATAFRAME)
        line_data = line_data[line_data.conf > 30].dropna(subset=['text'])

        recognized_lines = []
        image_pil = Image.open(image_path).convert("RGB")

        # 2. จัดกลุ่มคำเป็นบรรทัด แล้วอ่านทีละบรรทัดด้วย TrOCR
        for block_num in line_data['block_num'].unique():
            block_df = line_data[line_data['block_num'] == block_num]
            for par_num in block_df['par_num'].unique():
                par_df = block_df[block_df['par_num'] == par_num]
                for line_num in par_df['line_num'].unique():
                    line_df = par_df[par_df['line_num'] == line_num]

                    if not line_df.empty:
                        # หา Bounding Box ของทั้งบรรทัด
                        x = line_df.left.min()
                        y = line_df.top.min()
                        w = line_df.left.max() + line_df.width.iloc[-1] - x
                        h = line_df.top.max() + line_df.height.iloc[-1] - y

                        # ตัดภาพเฉพาะบรรทัด
                        line_image = image_pil.crop((x, y, x + w, y + h))

                        # ส่งภาพบรรทัดให้ TrOCR อ่าน
                        if line_image.size[0] > 10 and line_image.size[1] > 10: # กันภาพขนาดเล็กเกินไป
                            pixel_values = processor(images=line_image, return_tensors="pt").pixel_values.to(DEVICE)
                            generated_ids = model.generate(pixel_values, max_new_tokens=100)
                            generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
                            recognized_lines.append(generated_text)

        # 3. รวมข้อความที่ได้จาก TrOCR แล้วส่งให้ Parser
        full_text = "\n".join(recognized_lines)
        extracted_data = final_robust_parser(full_text)

    except Exception as ocr_error:
        print(f"Error processing file {filename}: {ocr_error}")
        extracted_data = {}

    results_list.append({
        "No.": image_id,
        "Application No. (GT)": gt_row["Application No."], "Application No. (Pred)": extracted_data.get('application_no'),
        "Name (GT)": gt_row["Name"], "Name (Pred)": extracted_data.get('name'),
        "Test Date (GT)": gt_row["Test Date"], "Test Date (Pred)": extracted_data.get('test_date'),
        "Grammar_Level (GT)": gt_row["Grammar_Level"], "Grammar_Level (Pred)": extracted_data.get('grammar_level'),
        "Grammar_Score (GT)": gt_row["Grammar_Score"], "Grammar_Score (Pred)": extracted_data.get('grammar_score'),
        "Reading_Level (GT)": gt_row["Reading_Level"], "Reading_Level (Pred)": extracted_data.get('reading_level'),
        "Reading_Score (GT)": gt_row["Reading_Score"], "Reading_Score (Pred)": extracted_data.get('reading_score'),
        "Speaking_Level (GT)": gt_row["Speaking_Level"], "Speaking_Level (Pred)": extracted_data.get('speaking_level'),
        "Speaking_Score (GT)": gt_row["Speaking_Score"], "Speaking_Score (Pred)": extracted_data.get('speaking_score'),
        "Writing_Level (GT)": gt_row["Writing_Level"], "Writing_Level (Pred)": extracted_data.get('writing_level'),
        "Writing_Score (GT)": gt_row["Writing_Score"], "Writing_Score (Pred)": extracted_data.get('writing_score'),
        "Total_Level (GT)": gt_row["Total_Level"], "Total_Level (Pred)": extracted_data.get('total_level'),
        "Total_Score (GT)": gt_row["Total_Score"], "Total_Score (Pred)": extracted_data.get('total_score'),
    })

ocr_results_df = pd.DataFrame(results_list)
print("--- ✅ สกัดข้อมูลเสร็จสิ้น ---\n")

# --- 5 & 6. การประเมินผลและบันทึก ---
# ... (วางโค้ดส่วนที่ 5 และ 6 ที่คุณมีต่อจากนี้ได้เลย) ...
# --- 5. เริ่มกระบวนการประเมินผล (Evaluation) ---
print("--- 📊 เริ่มการประเมินผล ---")
eval_df = ocr_results_df.fillna('')
fields_to_evaluate = {
    'Name': ('Name (GT)', 'Name (Pred)'), 'Application No.': ('Application No. (GT)', 'Application No. (Pred)'),
    'Test Date': ('Test Date (GT)', 'Test Date (Pred)'), 'Grammar_Level': ('Grammar_Level (GT)', 'Grammar_Level (Pred)'),
    'Grammar_Score': ('Grammar_Score (GT)', 'Grammar_Score (Pred)'), 'Reading_Level': ('Reading_Level (GT)', 'Reading_Level (Pred)'),
    'Reading_Score': ('Reading_Score (GT)', 'Reading_Score (Pred)'), 'Speaking_Level': ('Speaking_Level (GT)', 'Speaking_Level (Pred)'),
    'Speaking_Score': ('Speaking_Score (GT)', 'Speaking_Score (Pred)'), 'Writing_Level': ('Writing_Level (GT)', 'Writing_Level (Pred)'),
    'Writing_Score': ('Writing_Score (GT)', 'Writing_Score (Pred)'), 'Total_Level': ('Total_Level (GT)', 'Total_Level (Pred)'),
    'Total_Score': ('Total_Score (GT)', 'Total_Score (Pred)'),
}
evaluation_summary_list = []
for field, (gt_col, pred_col) in fields_to_evaluate.items():
    ground_truth = [normalize_text(t) for t in eval_df[gt_col]]
    prediction = [normalize_text(t) for t in eval_df[pred_col]]
    accuracy = np.mean([1 if gt == pred else 0 for gt, pred in zip(ground_truth, prediction)]) * 100
    try:
        error_metrics = jiwer.compute_measures(ground_truth, prediction)
        wer = error_metrics.get('wer', 0) * 100
        cer = error_metrics.get('cer', 0) * 100
        H = error_metrics.get('hits', 0); I = error_metrics.get('insertions', 0); D = error_metrics.get('deletions', 0); S = error_metrics.get('substitutions', 0)
        precision = H / (H + I + S) if (H + I + S) > 0 else 0
        recall = H / (H + D + S) if (H + D + S) > 0 else 0
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    except ValueError:
        wer, cer, f1_score = 100, 100, 0
    evaluation_summary_list.append({
        'Field': field, 'Accuracy (%)': round(accuracy, 2), 'WER (%)': round(wer, 2),
        'CER (%)': round(cer, 2), 'F1-score (%)': round(f1_score * 100, 2)
    })
eval_summary_df = pd.DataFrame(evaluation_summary_list)
eval_summary_df.insert(0, 'Approach', APPROACH_NAME)
eval_summary_df.insert(0, 'Model', MODEL_NAME)
print(eval_summary_df.to_string(index=False))
print("--- ✅ ประเมินผลเสร็จสิ้น ---\n")

# --- 6. บันทึกผลลัพธ์ ---
print(f"--- 💾 กำลังบันทึกรายงานเฉพาะของ {MODEL_NAME} ({APPROACH_NAME}) ---")
with pd.ExcelWriter(INDIVIDUAL_REPORT_PATH, engine='openpyxl') as writer:
    ocr_results_df.to_excel(writer, sheet_name='Detailed_Results', index=False)
    eval_summary_df.drop(columns=['Model', 'Approach']).to_excel(writer, sheet_name='Evaluation_Summary', index=False)
print(f"🎉 บันทึกไฟล์ {INDIVIDUAL_REPORT_PATH} สำเร็จ!")

print(f"--- 💾 กำลังอัปเดตไฟล์ Master Report: {MASTER_OUTPUT_PATH} ---")
SHEET_NAME = 'Master_Evaluation'
try:
    master_df = pd.read_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME)
    master_df = master_df[~((master_df['Model'] == MODEL_NAME) & (master_df['Approach'] == APPROACH_NAME))]
    combined_df = pd.concat([master_df, eval_summary_df], ignore_index=True)
except FileNotFoundError:
    print(f"ไม่พบไฟล์ Master เดิม, กำลังสร้างไฟล์ใหม่...")
    combined_df = eval_summary_df
combined_df.to_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME, index=False)
print(f"🎉 อัปเดตไฟล์ Master Report สำเร็จ!")

**Boxing**

In [ ]:
import cv2
import pandas as pd
import os
import re
import jiwer
import numpy as np
from tqdm import tqdm
from PIL import Image

# ⬇️⬇️⬇️ เพิ่ม import สำหรับ TrOCR ⬇️⬇️⬇️
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# --- 1. กำหนดค่าและ Path ที่สำคัญ ---
MODEL_NAME = 'TrOCR' # <<< เปลี่ยนชื่อโมเดล
APPROACH_NAME = 'Boxing OCR'
MASTER_OUTPUT_PATH = "/content/drive/MyDrive/lastest-ktep/excel/comparison_report.xlsx"
INDIVIDUAL_REPORT_PATH = f"/content/drive/MyDrive/lastest-ktep/excel/{MODEL_NAME.lower()}_{APPROACH_NAME.replace(' ', '_').lower()}_report.xlsx"
IMAGE_DIR = "/content/drive/MyDrive/lastest-ktep/prep-img"
GROUND_TRUTH_PATH = "/content/drive/MyDrive/lastest-ktep/excel/คะแนน TEP (Pilot Study)_Total_IT.xlsx"

# พิกัด ROI ที่แน่นอน
ROI_CONFIG = {
    'name': (200, 485, 600, 65), 'application_no': (1100, 485, 300, 60),
    'test_date': (1310, 545, 300, 60), 'overall_score': (820, 625, 800, 100),
    'grammar': (320, 785, 500, 80), 'reading': (1050, 785, 500, 80),
    'speaking': (320, 900, 500, 80), 'writing': (1050, 900, 500, 80),
}

# --- 2. ฟังก์ชันช่วยเหลือ (ไม่มีการเปลี่ยนแปลง) ---
def get_id_from_filename(filename):
    match = re.search(r'(\d+)\.(png|jpg|jpeg)$', filename.lower())
    if match: return int(match.group(1))
    return None

def normalize_text(text):
    if text is None: return ""
    text_str = str(text).strip().lower().replace(' ', '')
    if text_str.endswith('.0'): text_str = text_str[:-2]
    return text_str

def parse_level_and_score(text):
    if not text: return None, None
    cleaned_text = text.upper().replace('I', '1').replace('l', '1')
    match = re.search(r"([A-Z1-9]+)\s*\(\s*(\d+)\)", cleaned_text)
    if match:
        level, score = match.group(1), match.group(2)
        if len(level) > 2 and level.startswith('B'): level = 'B1'
        return level, score
    return None, None

# --- 3. โหลดโมเดลและข้อมูล ---
print("--- 🔍 ขั้นตอนโหลดโมเดลและข้อมูล ---")
# ⬇️⬇️⬇️ โหลดโมเดล TrOCR (Printed) ⬇️⬇️⬇️
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
trocr_processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
trocr_model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed').to(DEVICE)
print(f"TrOCR model loaded, using device: {DEVICE}")
# ⬆️⬆️⬆️ สิ้นสุดการโหลดโมเดล ⬆️⬆️⬆️

gt_df = pd.read_excel(GROUND_TRUTH_PATH, dtype={'Application No.': str})
gt_df['No.'] = gt_df['No.'].astype(int)
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))], key=lambda f: get_id_from_filename(f) or 0)
print(f"พบ ID ใน Ground Truth ทั้งหมด: {len(gt_df['No.'].unique())} ID | พบไฟล์รูปภาพ: {len(image_files)} ไฟล์")
print("--- ✅ ตรวจสอบข้อมูลเสร็จสิ้น ---\n")

# --- 4. เริ่มกระบวนการสกัดข้อมูล ---
results_list = []
print(f"--- 🚀 เริ่มการสกัดข้อมูลจากรูปภาพ {len(image_files)} ไฟล์ ด้วย {MODEL_NAME} ({APPROACH_NAME}) ---")
for filename in tqdm(image_files, desc=f"Processing with {MODEL_NAME}"):
    image_id = get_id_from_filename(filename)
    if image_id is None: continue

    gt_row_df = gt_df[gt_df['No.'] == image_id]
    if gt_row_df.empty: continue
    gt_row = gt_row_df.iloc[0]

    image_path = os.path.join(IMAGE_DIR, filename)

    try:
        processed_image_pil = Image.open(image_path)
        extracted_data = {}

        for field, (x, y, w, h) in ROI_CONFIG.items():
            x1, y1, x2, y2 = x, y, x + w, y + h
            cropped_box = processed_image_pil.crop((x1, y1, x2, y2))

            # ⬇️⬇️⬇️ เปลี่ยนมาใช้ TrOCR ในการสกัดข้อความ ⬇️⬇️⬇️
            # TrOCR ต้องการภาพสี RGB
            pixel_values = trocr_processor(images=cropped_box.convert("RGB"), return_tensors="pt").pixel_values.to(DEVICE)
            generated_ids = trocr_model.generate(pixel_values, max_new_tokens=50)
            text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
            # ⬆️⬆️⬆️ สิ้นสุดส่วนที่เปลี่ยน ⬆️⬆️⬆️

            if field in ['grammar', 'reading', 'speaking', 'writing', 'overall_score']:
                level, score = parse_level_and_score(text)
                if field == 'overall_score':
                    extracted_data['total_level'], extracted_data['total_score'] = level, score
                else:
                    extracted_data[f'{field}_level'], extracted_data[f'{field}_score'] = level, score
            else:
                extracted_data[field] = text
    except Exception as ocr_error:
        print(f"Error processing file {filename}: {ocr_error}")
        continue

    results_list.append({
        "No.": image_id,
        "Application No. (GT)": gt_row["Application No."], "Application No. (Pred)": extracted_data.get('application_no'),
        "Name (GT)": gt_row["Name"], "Name (Pred)": extracted_data.get('name'),
        "Test Date (GT)": gt_row["Test Date"], "Test Date (Pred)": extracted_data.get('test_date'),
        "Grammar_Level (GT)": gt_row["Grammar_Level"], "Grammar_Level (Pred)": extracted_data.get('grammar_level'),
        "Grammar_Score (GT)": gt_row["Grammar_Score"], "Grammar_Score (Pred)": extracted_data.get('grammar_score'),
        "Reading_Level (GT)": gt_row["Reading_Level"], "Reading_Level (Pred)": extracted_data.get('reading_level'),
        "Reading_Score (GT)": gt_row["Reading_Score"], "Reading_Score (Pred)": extracted_data.get('reading_score'),
        "Speaking_Level (GT)": gt_row["Speaking_Level"], "Speaking_Level (Pred)": extracted_data.get('speaking_level'),
        "Speaking_Score (GT)": gt_row["Speaking_Score"], "Speaking_Score (Pred)": extracted_data.get('speaking_score'),
        "Writing_Level (GT)": gt_row["Writing_Level"], "Writing_Level (Pred)": extracted_data.get('writing_level'),
        "Writing_Score (GT)": gt_row["Writing_Score"], "Writing_Score (Pred)": extracted_data.get('writing_score'),
        "Total_Level (GT)": gt_row["Total_Level"], "Total_Level (Pred)": extracted_data.get('total_level'),
        "Total_Score (GT)": gt_row["Total_Score"], "Total_Score (Pred)": extracted_data.get('total_score'),
    })

ocr_results_df = pd.DataFrame(results_list)
print("--- ✅ สกัดข้อมูลเสร็จสิ้น ---\n")

# --- 5. เริ่มกระบวนการประเมินผล (Evaluation) ---
print("--- 📊 เริ่มการประเมินผล ---")
eval_df = ocr_results_df.fillna('')
fields_to_evaluate = {
    'Name': ('Name (GT)', 'Name (Pred)'), 'Application No.': ('Application No. (GT)', 'Application No. (Pred)'),
    'Test Date': ('Test Date (GT)', 'Test Date (Pred)'), 'Grammar_Level': ('Grammar_Level (GT)', 'Grammar_Level (Pred)'),
    'Grammar_Score': ('Grammar_Score (GT)', 'Grammar_Score (Pred)'), 'Reading_Level': ('Reading_Level (GT)', 'Reading_Level (Pred)'),
    'Reading_Score': ('Reading_Score (GT)', 'Reading_Score (Pred)'), 'Speaking_Level': ('Speaking_Level (GT)', 'Speaking_Level (Pred)'),
    'Speaking_Score': ('Speaking_Score (GT)', 'Speaking_Score (Pred)'), 'Writing_Level': ('Writing_Level (GT)', 'Writing_Level (Pred)'),
    'Writing_Score': ('Writing_Score (GT)', 'Writing_Score (Pred)'), 'Total_Level': ('Total_Level (GT)', 'Total_Level (Pred)'),
    'Total_Score': ('Total_Score (GT)', 'Total_Score (Pred)'),
}
evaluation_summary_list = []
for field, (gt_col, pred_col) in fields_to_evaluate.items():
    ground_truth = [normalize_text(t) for t in eval_df[gt_col]]
    prediction = [normalize_text(t) for t in eval_df[pred_col]]
    accuracy = np.mean([1 if gt == pred else 0 for gt, pred in zip(ground_truth, prediction)]) * 100
    try:
        error_metrics = jiwer.compute_measures(ground_truth, prediction)
        wer = error_metrics.get('wer', 0) * 100
        cer = error_metrics.get('cer', 0) * 100
        H = error_metrics.get('hits', 0); I = error_metrics.get('insertions', 0); D = error_metrics.get('deletions', 0); S = error_metrics.get('substitutions', 0)
        precision = H / (H + I + S) if (H + I + S) > 0 else 0
        recall = H / (H + D + S) if (H + D + S) > 0 else 0
        f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    except ValueError:
        wer, cer, f1_score = 100, 100, 0
    evaluation_summary_list.append({
        'Field': field, 'Accuracy (%)': round(accuracy, 2), 'WER (%)': round(wer, 2),
        'CER (%)': round(cer, 2), 'F1-score (%)': round(f1_score * 100, 2)
    })
eval_summary_df = pd.DataFrame(evaluation_summary_list)
eval_summary_df.insert(0, 'Approach', APPROACH_NAME)
eval_summary_df.insert(0, 'Model', MODEL_NAME)
print(eval_summary_df.to_string(index=False))
print("--- ✅ ประเมินผลเสร็จสิ้น ---\n")

# --- 6. บันทึกผลลัพธ์ ---
print(f"--- 💾 กำลังบันทึกรายงานเฉพาะของ {MODEL_NAME} ({APPROACH_NAME}) ---")
with pd.ExcelWriter(INDIVIDUAL_REPORT_PATH, engine='openpyxl') as writer:
    ocr_results_df.to_excel(writer, sheet_name='Detailed_Results', index=False)
    eval_summary_df.drop(columns=['Model', 'Approach']).to_excel(writer, sheet_name='Evaluation_Summary', index=False)
print(f"🎉 บันทึกไฟล์ {INDIVIDUAL_REPORT_PATH} สำเร็จ!")

print(f"--- 💾 กำลังอัปเดตไฟล์ Master Report: {MASTER_OUTPUT_PATH} ---")
SHEET_NAME = 'Master_Evaluation'
try:
    master_df = pd.read_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME)
    master_df = master_df[~((master_df['Model'] == MODEL_NAME) & (master_df['Approach'] == APPROACH_NAME))]
    combined_df = pd.concat([master_df, eval_summary_df], ignore_index=True)
except FileNotFoundError:
    print(f"ไม่พบไฟล์ Master เดิม, กำลังสร้างไฟล์ใหม่...")
    combined_df = eval_summary_df
combined_df.to_excel(MASTER_OUTPUT_PATH, sheet_name=SHEET_NAME, index=False)
print(f"🎉 อัปเดตไฟล์ Master Report สำเร็จ!")

# Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. กำหนดค่าและ Path ---
MASTER_REPORT_PATH = "/content/drive/MyDrive/lastest-ktep/excel/comparison_report.xlsx"
SHEET_NAME = 'Master_Evaluation'

# --- 2. โหลดและเตรียมข้อมูล ---
try:
    print(f"--- 🔍 กำลังโหลดข้อมูลจาก Master Report: {MASTER_REPORT_PATH} ---")
    df = pd.read_excel(MASTER_REPORT_PATH, sheet_name=SHEET_NAME)
    print("--- ✅ โหลดข้อมูลสำเร็จ ---")

    # สร้างคอลัมน์สำหรับใช้ในกราฟที่เข้าใจง่าย
    df['Model_Approach'] = df['Model'] + ' (' + df['Approach'] + ')'

    # กรองเอาเฉพาะแนวทางที่ใช้งานได้จริง (ตัด TrOCR Pure OCR ที่ไม่สมบูรณ์ออก)
    df = df[~df['Model_Approach'].str.contains('TrOCR \(Pure OCR', na=False)]

except FileNotFoundError:
    print(f"[ERROR] ไม่พบไฟล์ Master Report ที่: {MASTER_REPORT_PATH}")
    # จบการทำงานหากไม่มีไฟล์
    exit()
except Exception as e:
    print(f"เกิดข้อผิดพลาดในการอ่านไฟล์: {e}")
    exit()

# --- 3. คำนวณและแสดงค่าเฉลี่ย ---
print("\n" + "="*60)
print("📊 ตารางสรุปค่าเฉลี่ยประสิทธิภาพ (Average Performance) 📊")
print("="*60)

# คำนวณค่าเฉลี่ยโดยจัดกลุ่มตาม Model และ Approach
average_performance = df.groupby(['Model', 'Approach']).mean(numeric_only=True).round(2)

# แสดงผลเฉพาะคอลัมน์ที่สำคัญ
display(average_performance[['Accuracy (%)', 'WER (%)', 'CER (%)', 'F1-score (%)']])


# --- 4. สร้างกราฟเปรียบเทียบ (Visualization) ---
print("\n" + "="*60)
print("📈 กราฟเปรียบเทียบประสิทธิภาพ 📈")
print("="*60)

# ตั้งค่า Style ของกราฟให้อ่านง่าย
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (14, 8)

# --- กราฟที่ 1: เปรียบเทียบภาพรวมทั้งหมด (หาผู้ชนะ) ---
# F1-score เป็นตัวชี้วัดโดยรวมที่ดีที่สุด
plt.figure() # สร้าง figure ใหม่
overall_avg_f1 = df.groupby('Model_Approach')['F1-score (%)'].mean().sort_values(ascending=False)
ax1 = sns.barplot(x=overall_avg_f1.values, y=overall_avg_f1.index, palette='plasma', orient='h')
ax1.set_title('Overall Performance Ranking (based on Average F1-score)', fontsize=18, pad=20)
ax1.set_xlabel('Average F1-score (%)', fontsize=14)
ax1.set_ylabel('Model (Approach)', fontsize=14)
# แสดงตัวเลขกำกับบนแท่งกราฟ
for i, v in enumerate(overall_avg_f1.values):
    ax1.text(v + 0.5, i, f'{v:.2f}', color='black', va='center')
plt.tight_layout()
plt.show()


# --- กราฟที่ 2: เปรียบเทียบโมเดลในแนวทาง "Boxing OCR" ---
plt.figure() # สร้าง figure ใหม่
df_boxing = df[df['Approach'] == 'Boxing OCR']
ax2 = sns.barplot(data=df_boxing, x='Model', y='Accuracy (%)', palette='viridis')
ax2.set_title('Model Comparison for "Boxing OCR" (Average Accuracy)', fontsize=18, pad=20)
ax2.set_xlabel('Model', fontsize=14)
ax2.set_ylabel('Average Accuracy (%)', fontsize=14)
ax2.bar_label(ax2.containers[0], fmt='%.2f')
plt.ylim(0, 110) # กำหนดให้แกน Y สูงสุดที่ 110 เพื่อให้เห็นความแตกต่างชัดเจน
plt.tight_layout()
plt.show()


# --- กราฟที่ 3: เปรียบเทียบโมเดลในแนวทาง "Pure OCR" ---
plt.figure() # สร้าง figure ใหม่
df_pure = df[df['Approach'].str.contains('Pure OCR', na=False)]
ax3 = sns.barplot(data=df_pure, x='Model', y='Accuracy (%)', palette='coolwarm')
ax3.set_title('Model Comparison for "Pure OCR" (Average Accuracy)', fontsize=18, pad=20)
ax3.set_xlabel('Model', fontsize=14)
ax3.set_ylabel('Average Accuracy (%)', fontsize=14)
ax3.bar_label(ax3.containers[0], fmt='%.2f')
plt.ylim(0, 110)
plt.tight_layout()
plt.show()


# --- 5. สรุปผลในฐานะผู้เชี่ยวชาญ ---
print("\n" + "="*60)
print("🏆 บทสรุปและข้อเสนอแนะสุดท้าย (Final Conclusion) 🏆")
print("="*60)

# หา "Winning Combination" จาก F1-score ที่สูงสุด
winner = overall_avg_f1.index[0]
winner_score = overall_avg_f1.values[0]

print(f"จากการทดลองทั้งหมด สามารถสรุปได้อย่างชัดเจนว่า:\n")
print(f"แนวทางและโมเดลที่ให้ประสิทธิภาพสูงสุด (The Winning Combination) คือ: '{winner}'")
print(f"ด้วยค่าเฉลี่ย F1-score สูงถึง {winner_score:.2f} %\n")
print("เหตุผลสนับสนุน:")
print("1. แนวทาง 'Boxing OCR' ให้ผลลัพธ์ที่แม่นยำและเสถียรกว่า 'Pure OCR' อย่างมีนัยสำคัญในทุกโมเดล")
print("   เนื่องจากไม่ได้รับผลกระทบจากการที่ OCR อ่านคีย์เวิร์ดหรือโครงสร้างเอกสารผิดพลาด")
print("2. ภายในกลุ่ม Boxing OCR, โมเดล 'TrOCR' แสดงให้เห็นถึงความสามารถในการอ่านตัวอักษรและตัวเลขที่ถูกตัดมาแล้ว")
print("   ได้อย่างแม่นยำที่สุด ซึ่งสอดคล้องกับสถาปัตยกรรมของโมเดลที่ถูกออกแบบมาเพื่องานลักษณะนี้โดยเฉพาะ\n")
print("ดังนั้น สำหรับโปรเจกต์ที่ต้องการความถูกต้องสูงสุดเพื่อนำไปใช้งานจริง ขอแนะนำให้ใช้ 'TrOCR (Boxing OCR)' เป็นวิธีหลักครับ")
print("\n--- 🏁 สิ้นสุดการวิเคราะห์โปรเจกต์ 🏁 ---")